# FIFA World Cup 2026 — Goals Collection

This notebook collects real FIFA World Cup 2026 goal-event data.

The final dataset will be used to populate the Goals SQL table.

Each goal record will contain:

- GoalID
- MatchID
- PlayerID
- TeamID
- Minute
- GoalType

The data will be collected from a real World Cup 2026 data source
and matched to our existing Teams, Players, and Matches datasets.

No fabricated or random goal data will be used.

In [1]:
from pathlib import Path
import pandas as pd
import requests

data_folder = Path.cwd().parent / "Data"

print("PROJECT SETUP")
print("=" * 80)

print("Data folder:", data_folder)
print("Data folder exists:", data_folder.exists())

print("\nRequired files:")

required_files = [
    "Teams.csv",
    "Players.csv",
    "Matches.csv"
]

for file_name in required_files:
    file_path = data_folder / file_name
    print(
        file_name,
        "->",
        "Found" if file_path.exists() else "MISSING"
    )

PROJECT SETUP
Data folder: d:\ai course hw\FIFA_WorldCup_2026_Project\Data
Data folder exists: True

Required files:
Teams.csv -> Found
Players.csv -> Found
Matches.csv -> Found


## 2. Test the Goal Data Source

We will test the detailed match endpoint for one World Cup 2026
match.

The response should contain the match information and its goal
events.

We will inspect the goal structure before collecting all 104 matches.

In [2]:
import requests

test_match_id = 0

url = (
    "https://foot-boll.com/api/data.php"
    f"?action=match&id={test_match_id}"
)

response = requests.get(url, timeout=30)

print("HTTP Status:", response.status_code)
print("Content Type:", response.headers.get("Content-Type"))

response.raise_for_status()

match_data = response.json()

print("\nResponse Type:", type(match_data).__name__)
print("Top-level Keys:", list(match_data.keys()))

print("\nMatch keys:")
print(list(match_data["match"].keys()))

HTTP Status: 200
Content Type: application/json; charset=utf-8

Response Type: dict
Top-level Keys: ['ok', 'action', 'updated', 'match']

Match keys:
['id', 'round', 'group', 'team1', 'team2', 'team1_ar', 'team2_ar', 'flag1', 'flag2', 'status', 'score', 'penalties', 'winner', 'live_minute', 'date', 'time', 'datetime', 'ground', 'ht', 'goals1', 'goals2', 'cards', 'stats']


## 3. Inspect Goal Event Structure

The detailed match response contains `goals1` and `goals2`.

We will inspect these fields to determine:

- How individual goals are represented.
- Which player scored.
- How the goal minute is stored.
- Whether the source provides a goal type.
- How own goals and other goal types are represented.

We will use this structure to design the final Goals dataset.

In [3]:
match = match_data["match"]

print("GOALS 1")
print("=" * 80)
print(match["goals1"])

print("\nGOALS 2")
print("=" * 80)
print(match["goals2"])

GOALS 1
[{'name': 'Julián Quiñones', 'minute': '9'}, {'name': 'Raúl Jiménez', 'minute': '67'}]

GOALS 2
[]


## 4. Inspect Goal Type Information

The goal records currently provide the scorer name and minute.

Our SQL design also includes GoalType.

We will inspect the complete goal-related information in the
match response to determine whether the source provides
additional information such as:

- Normal goal
- Own goal
- Penalty
- Other goal types

We will not create or assume goal types that are not present
in the source.

In [4]:
print("GOAL-RELATED FIELDS")
print("=" * 80)

for key, value in match.items():
    key_lower = key.lower()

    if (
        "goal" in key_lower
        or "score" in key_lower
        or "penalt" in key_lower
    ):
        print(f"\n{key}:")
        print(value)

GOAL-RELATED FIELDS

score:
[2, 0]

penalties:
None

goals1:
[{'name': 'Julián Quiñones', 'minute': '9'}, {'name': 'Raúl Jiménez', 'minute': '67'}]

goals2:
[]


## 6. Load Reference Datasets

We will load the existing Teams, Players, and Matches datasets.

These datasets provide the IDs required by the Goals table:

- TeamID
- PlayerID
- MatchID

The goal source provides names, so we will match those names
to our validated database IDs rather than creating new IDs.

In [5]:
teams_df = pd.read_csv(data_folder / "Teams.csv")
players_df = pd.read_csv(data_folder / "Players.csv")
matches_df = pd.read_csv(data_folder / "Matches.csv")

print("REFERENCE DATA")
print("=" * 80)

print("Teams:", teams_df.shape)
print("Players:", players_df.shape)
print("Matches:", matches_df.shape)

print("\nTeam columns:")
print(teams_df.columns.tolist())

print("\nPlayer columns:")
print(players_df.columns.tolist())

print("\nMatch columns:")
print(matches_df.columns.tolist())

REFERENCE DATA
Teams: (48, 6)
Players: (1248, 5)
Matches: (104, 11)

Team columns:
['TeamID', 'TeamName', 'CountryCode', 'Confederation', 'GroupName', 'HostCountry']

Player columns:
['PlayerID', 'PlayerName', 'TeamID', 'Position', 'JerseyNumber']

Match columns:
['MatchID', 'MatchDate', 'Stage', 'GroupName', 'StadiumID', 'Team1ID', 'Team2ID', 'Team1Goals', 'Team2Goals', 'WinnerTeamID', 'Result']


## 7. Create Team Name Mapping

The goal source identifies teams by name.

Our database identifies teams using TeamID.

We will create a TeamName-to-TeamID mapping and compare the
source team names with our validated Teams dataset.

Any naming differences will be identified before we collect
the goal records.

In [6]:
team_name_to_id = dict(
    zip(
        teams_df["TeamName"],
        teams_df["TeamID"]
    )
)

print("TEAM MAPPING")
print("=" * 80)

print("Number of teams in mapping:", len(team_name_to_id))

print("\nFirst 10 mappings:")

for team_name, team_id in list(team_name_to_id.items())[:10]:
    print(f"{team_name} -> TeamID {team_id}")

TEAM MAPPING
Number of teams in mapping: 48

First 10 mappings:
Argentina -> TeamID 1
Belgium -> TeamID 2
Brazil -> TeamID 3
Colombia -> TeamID 4
England -> TeamID 5
France -> TeamID 6
Germany -> TeamID 7
Mexico -> TeamID 8
Netherlands -> TeamID 9
Spain -> TeamID 10


## 8. Inspect Source Team Names

The source may use slightly different names for some teams.

We will collect all unique team names appearing in the
104 detailed match records and compare them with our
Teams.csv names.

This allows us to identify every required name alias before
creating the final Goal records.

In [7]:
all_matches_response = requests.get(
    "https://foot-boll.com/api/data.php?action=all",
    timeout=30
)

all_matches_response.raise_for_status()

all_matches_data = all_matches_response.json()["matches"]

source_team_names = set()

for match_item in all_matches_data:
    source_team_names.add(match_item["team1"])
    source_team_names.add(match_item["team2"])

database_team_names = set(teams_df["TeamName"])

print("SOURCE TEAM NAME CHECK")
print("=" * 80)

print("Source unique team names:", len(source_team_names))
print("Database team names:", len(database_team_names))

print("\nNames not found directly in Teams.csv:")

unmatched_names = sorted(
    source_team_names - database_team_names
)

for name in unmatched_names:
    print("-", name)

print("\nTotal unmatched names:", len(unmatched_names))

SOURCE TEAM NAME CHECK
Source unique team names: 48
Database team names: 48

Names not found directly in Teams.csv:
- Bosnia & Herzegovina
- Cape Verde
- Czech Republic
- DR Congo
- Iran
- Ivory Coast
- South Korea
- Turkey

Total unmatched names: 8


## 9. Normalize Team Names

The goal source uses eight team names that differ from the
official team names in our Teams dataset.

We will apply the verified aliases so every source team can
be mapped to its existing TeamID.

No new teams will be created.

In [8]:
team_aliases = {
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "Cape Verde": "Cabo Verde",
    "Czech Republic": "Czechia",
    "DR Congo": "Congo DR",
    "Iran": "IR Iran",
    "Ivory Coast": "Côte d'Ivoire",
    "South Korea": "Korea Republic",
    "Turkey": "Türkiye"
}

normalized_source_names = {
    name: team_aliases.get(name, name)
    for name in source_team_names
}

unresolved_names = [
    source_name
    for source_name, database_name in normalized_source_names.items()
    if database_name not in team_name_to_id
]

print("TEAM ALIAS VALIDATION")
print("=" * 80)

print("Aliases defined:", len(team_aliases))
print("Unresolved team names:", len(unresolved_names))

if unresolved_names:
    print("\nUnresolved names:")
    for name in unresolved_names:
        print("-", name)
else:
    print("\nAll 48 source team names can now be mapped to TeamID.")

print("\nAlias mappings:")

for source_name, database_name in team_aliases.items():
    print(
        f"{source_name} -> {database_name} "
        f"-> TeamID {team_name_to_id[database_name]}"
    )

TEAM ALIAS VALIDATION
Aliases defined: 8
Unresolved team names: 0

All 48 source team names can now be mapped to TeamID.

Alias mappings:
Bosnia & Herzegovina -> Bosnia and Herzegovina -> TeamID 26
Cape Verde -> Cabo Verde -> TeamID 15
Czech Republic -> Czechia -> TeamID 38
DR Congo -> Congo DR -> TeamID 27
Iran -> IR Iran -> TeamID 30
Ivory Coast -> Côte d'Ivoire -> TeamID 17
South Korea -> Korea Republic -> TeamID 31
Turkey -> Türkiye -> TeamID 47


## 10. Build Source Match to Database Match Mapping

The source match IDs do not necessarily correspond to our
internal MatchIDs.

We will identify each match using the two participating teams.

The team order will be ignored, so:

Team A vs Team B

and

Team B vs Team A

will be treated as the same fixture.

This will allow every source goal event to be assigned to
the correct MatchID.

In [9]:
# Create a lookup using the two TeamIDs in each fixture.
# Sorting the IDs makes the mapping independent of team order.

match_key_to_id = {}

for _, row in matches_df.iterrows():
    team_ids = tuple(
        sorted([
            int(row["Team1ID"]),
            int(row["Team2ID"])
        ])
    )

    if team_ids in match_key_to_id:
        raise ValueError(
            f"Duplicate match key found: {team_ids}"
        )

    match_key_to_id[team_ids] = int(row["MatchID"])

print("MATCH MAPPING")
print("=" * 80)

print("Database matches:", len(matches_df))
print("Unique match keys:", len(match_key_to_id))

print("\nFirst 10 mappings:")

for key, match_id in list(match_key_to_id.items())[:10]:
    print(
        f"TeamIDs {key} -> MatchID {match_id}"
    )

MATCH MAPPING
Database matches: 104
Unique match keys: 104

First 10 mappings:
TeamIDs (8, 24) -> MatchID 1
TeamIDs (31, 38) -> MatchID 2
TeamIDs (16, 26) -> MatchID 3
TeamIDs (12, 32) -> MatchID 4
TeamIDs (33, 39) -> MatchID 5
TeamIDs (13, 47) -> MatchID 6
TeamIDs (3, 21) -> MatchID 7
TeamIDs (11, 44) -> MatchID 8
TeamIDs (17, 28) -> MatchID 9
TeamIDs (7, 37) -> MatchID 10


## 11. Collect Goal Events from All 104 Matches

We will request the detailed match endpoint for every World Cup 2026
match in the source.

For each match, we will collect:

- Source match ID
- Team 1
- Team 2
- Goals scored by Team 1
- Goals scored by Team 2

The raw goal events will be preserved so they can later be matched
to our internal MatchID, TeamID, and PlayerID values.

In [10]:
goal_match_records = []

for index, match_item in enumerate(all_matches_data, start=1):

    source_match_id = match_item["id"]

    url = (
        "https://foot-boll.com/api/data.php"
        f"?action=match&id={source_match_id}"
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    detailed_match = response.json()["match"]

    goal_match_records.append({
        "SourceMatchID": source_match_id,
        "Team1": detailed_match["team1"],
        "Team2": detailed_match["team2"],
        "Goals1": detailed_match["goals1"],
        "Goals2": detailed_match["goals2"]
    })

    if index % 20 == 0 or index == len(all_matches_data):
        print(
            f"Collected {index}/{len(all_matches_data)} matches"
        )

print("\nCOLLECTION COMPLETE")
print("=" * 80)

print("Total detailed matches:", len(goal_match_records))

Collected 20/104 matches
Collected 40/104 matches
Collected 60/104 matches
Collected 80/104 matches
Collected 100/104 matches
Collected 104/104 matches

COLLECTION COMPLETE
Total detailed matches: 104


## 12. Validate the Collected Goal Events

We will count all goal events collected from the 104 matches.

The total should match the World Cup 2026 tournament goal count
already established from our match dataset.

We will also inspect several raw goal records to confirm that
the scorer names and minutes were collected correctly.

In [11]:
total_goals = 0

for record in goal_match_records:
    total_goals += len(record["Goals1"])
    total_goals += len(record["Goals2"])

print("GOAL EVENT VALIDATION")
print("=" * 80)

print("Matches collected:", len(goal_match_records))
print("Total goal events:", total_goals)

print("\nExpected tournament goals:", 308)
print("Goal count matches expected:", total_goals == 308)

print("\nSample goal records:")

shown = 0

for record in goal_match_records:
    all_goals = record["Goals1"] + record["Goals2"]

    if all_goals:
        print(
            f"\nSourceMatchID: {record['SourceMatchID']}"
        )
        print(
            f"{record['Team1']} vs {record['Team2']}"
        )
        print("Goals:", all_goals)

        shown += 1

        if shown == 5:
            break

GOAL EVENT VALIDATION
Matches collected: 104
Total goal events: 308

Expected tournament goals: 308
Goal count matches expected: True

Sample goal records:

SourceMatchID: 0
Mexico vs South Africa
Goals: [{'name': 'Julián Quiñones', 'minute': '9'}, {'name': 'Raúl Jiménez', 'minute': '67'}]

SourceMatchID: 1
South Korea vs Czech Republic
Goals: [{'name': 'Hwang In-Beom', 'minute': '67'}, {'name': 'Oh Hyeon-Gyu', 'minute': '80'}, {'name': 'Ladislav Krejcí', 'minute': '59'}]

SourceMatchID: 2
Czech Republic vs South Africa
Goals: [{'name': 'Michal Sadílek', 'minute': '6'}, {'name': 'Teboho Mokoena', 'minute': '83', 'penalty': True}]

SourceMatchID: 3
Mexico vs South Korea
Goals: [{'name': 'Luis Romo', 'minute': '50'}]

SourceMatchID: 4
Czech Republic vs Mexico
Goals: [{'name': 'Mateo Chávez', 'minute': '55'}, {'name': 'Julián Quiñones', 'minute': '61'}, {'name': 'Álvaro Fidalgo', 'minute': '90+4'}]


## 13. Inspect Goal Record Fields

We will inspect every goal event collected from the 104 matches
and identify all fields used by the source.

This will allow us to correctly map:

- Player name
- Goal minute
- Penalty information
- Any other goal-related fields

We will only use information explicitly provided by the source.

In [12]:
goal_field_names = set()

for record in goal_match_records:
    for goal in record["Goals1"]:
        goal_field_names.update(goal.keys())

    for goal in record["Goals2"]:
        goal_field_names.update(goal.keys())

print("GOAL RECORD FIELD VALIDATION")
print("=" * 80)

print("Fields found:", len(goal_field_names))

for field in sorted(goal_field_names):
    print("-", field)

GOAL RECORD FIELD VALIDATION
Fields found: 4
- minute
- name
- owngoal
- penalty


## 14. Validate Penalty and Own-Goal Flags

Some goal records contain `penalty` or `owngoal`.

We will inspect their values across all 308 goal events.

This confirms exactly how the source represents:

- Penalty goals
- Own goals
- Normal goals

We will use only explicit True values when assigning GoalType.

In [13]:
all_goal_events = []

for record in goal_match_records:

    for goal in record["Goals1"]:
        all_goal_events.append(goal)

    for goal in record["Goals2"]:
        all_goal_events.append(goal)

penalty_goals = [
    goal for goal in all_goal_events
    if goal.get("penalty") is True
]

own_goals = [
    goal for goal in all_goal_events
    if goal.get("owngoal") is True
]

print("GOAL FLAG VALIDATION")
print("=" * 80)

print("Total goals:", len(all_goal_events))
print("Penalty goals:", len(penalty_goals))
print("Own goals:", len(own_goals))

print("\nPenalty goal examples:")
for goal in penalty_goals[:10]:
    print(goal)

print("\nOwn goal examples:")
for goal in own_goals[:10]:
    print(goal)

GOAL FLAG VALIDATION
Total goals: 308
Penalty goals: 16
Own goals: 14

Penalty goal examples:
{'name': 'Teboho Mokoena', 'minute': '83', 'penalty': True}
{'name': 'Breel Embolo', 'minute': '17', 'penalty': True}
{'name': 'Granit Xhaka', 'minute': '90+7', 'penalty': True}
{'name': 'Kai Havertz', 'minute': '45+5', 'penalty': True}
{'name': 'Marko Arnautovic', 'minute': '90+12', 'penalty': True}
{'name': 'Lautaro Martínez', 'minute': '31', 'penalty': True}
{'name': 'Yoane Wissa', 'minute': '68', 'penalty': True}
{'name': 'Harry Kane', 'minute': '12', 'penalty': True}
{'name': 'Youri Tielemans', 'minute': '120+5', 'penalty': True}
{'name': 'Cristiano Ronaldo', 'minute': '68', 'penalty': True}

Own goal examples:
{'name': 'Miro Muheim', 'minute': '90+4', 'owngoal': True}
{'name': 'Mohamed Manai', 'minute': '75', 'owngoal': True}
{'name': 'Mahmoud Abunada', 'minute': '34', 'owngoal': True}
{'name': 'Yassine Bounou', 'minute': '10', 'owngoal': True}
{'name': 'Damian Bobadilla', 'minute': '7',

## 16. Map Source Matches to Database MatchID

Each source match will be matched to our internal MatchID
using the two participating teams.

Team names will first be normalized using the verified aliases.

The mapping will be validated to ensure that all 104 source
matches correspond to exactly one database match.

In [14]:
source_match_mapping = []

for record in goal_match_records:

    team1_normalized = team_aliases.get(
        record["Team1"],
        record["Team1"]
    )

    team2_normalized = team_aliases.get(
        record["Team2"],
        record["Team2"]
    )

    team1_id = team_name_to_id[team1_normalized]
    team2_id = team_name_to_id[team2_normalized]

    match_key = tuple(sorted([team1_id, team2_id]))

    match_id = match_key_to_id.get(match_key)

    source_match_mapping.append({
        "SourceMatchID": record["SourceMatchID"],
        "MatchID": match_id,
        "Team1": record["Team1"],
        "Team2": record["Team2"],
        "Team1ID": team1_id,
        "Team2ID": team2_id
    })

mapping_df = pd.DataFrame(source_match_mapping)

print("SOURCE MATCH MAPPING")
print("=" * 80)

print("Source matches:", len(mapping_df))
print("Mapped MatchIDs:", mapping_df["MatchID"].notna().sum())
print(
    "Unmatched:",
    mapping_df["MatchID"].isna().sum()
)
print(
    "Duplicate MatchIDs:",
    mapping_df["MatchID"].duplicated().sum()
)

print("\nFirst 10 mappings:")
print(
    mapping_df.head(10).to_string(index=False)
)

SOURCE MATCH MAPPING
Source matches: 104
Mapped MatchIDs: 104
Unmatched: 0
Duplicate MatchIDs: 0

First 10 mappings:
 SourceMatchID  MatchID          Team1                Team2  Team1ID  Team2ID
             0        1         Mexico         South Africa        8       24
             1        2    South Korea       Czech Republic       31       38
             2       25 Czech Republic         South Africa       38       24
             3       28         Mexico          South Korea        8       31
             4       53 Czech Republic               Mexico       38        8
             5       54   South Africa          South Korea       24       31
             6        3         Canada Bosnia & Herzegovina       16       26
             7        8          Qatar          Switzerland       44       11
             8       26    Switzerland Bosnia & Herzegovina       11       26
             9       27         Canada                Qatar       16       44


## 17. Build the Raw Goal Dataset

We will transform the goal events from all 104 matches into
one row per goal.

Each goal will be associated with:

- Our internal MatchID
- The scoring TeamID
- The source player name
- The original goal minute
- GoalType when explicitly provided by the source

GoalType will not be inferred when the source does not provide it.

In [15]:
raw_goals = []

for record in goal_match_records:

    # Find our internal MatchID
    mapping_row = mapping_df[
        mapping_df["SourceMatchID"] == record["SourceMatchID"]
    ].iloc[0]

    match_id = int(mapping_row["MatchID"])

    team1_id = int(mapping_row["Team1ID"])
    team2_id = int(mapping_row["Team2ID"])

    # Goals scored by Team 1
    for goal in record["Goals1"]:

        if goal.get("penalty") is True:
            goal_type = "Penalty"
        elif goal.get("owngoal") is True:
            goal_type = "Own Goal"
        else:
            goal_type = None

        raw_goals.append({
            "SourceMatchID": record["SourceMatchID"],
            "MatchID": match_id,
            "TeamID": team1_id,
            "PlayerName": goal["name"],
            "Minute": str(goal["minute"]),
            "GoalType": goal_type
        })

    # Goals scored by Team 2
    for goal in record["Goals2"]:

        if goal.get("penalty") is True:
            goal_type = "Penalty"
        elif goal.get("owngoal") is True:
            goal_type = "Own Goal"
        else:
            goal_type = None

        raw_goals.append({
            "SourceMatchID": record["SourceMatchID"],
            "MatchID": match_id,
            "TeamID": team2_id,
            "PlayerName": goal["name"],
            "Minute": str(goal["minute"]),
            "GoalType": goal_type
        })

goals_raw_df = pd.DataFrame(raw_goals)

print("RAW GOAL DATASET")
print("=" * 80)

print("Rows:", len(goals_raw_df))
print("Columns:", len(goals_raw_df.columns))

print("\nColumns:")
print(goals_raw_df.columns.tolist())

print("\nGoal types:")
print(goals_raw_df["GoalType"].value_counts(dropna=False))

print("\nFirst 10 rows:")
print(goals_raw_df.head(10).to_string(index=False))

RAW GOAL DATASET
Rows: 308
Columns: 6

Columns:
['SourceMatchID', 'MatchID', 'TeamID', 'PlayerName', 'Minute', 'GoalType']

Goal types:
GoalType
None        278
Penalty      16
Own Goal     14
Name: count, dtype: int64

First 10 rows:
 SourceMatchID  MatchID  TeamID      PlayerName Minute GoalType
             0        1       8 Julián Quiñones      9     None
             0        1       8    Raúl Jiménez     67     None
             1        2      31   Hwang In-Beom     67     None
             1        2      31    Oh Hyeon-Gyu     80     None
             1        2      38 Ladislav Krejcí     59     None
             2       25      38  Michal Sadílek      6     None
             2       25      24  Teboho Mokoena     83  Penalty
             3       28       8       Luis Romo     50     None
             4       53       8    Mateo Chávez     55     None
             4       53       8 Julián Quiñones     61     None


## 18. Match Goal Scorers to PlayerID

The Players dataset contains the official PlayerID values.

The goal source provides player names and the scoring TeamID.

We will match each goal scorer using:

PlayerName + TeamID

This prevents players with the same name from being assigned
to the wrong PlayerID.

Before creating the final Goals table, we will identify any
unmatched player names.

In [16]:
player_lookup = (
    players_df
    .set_index(["PlayerName", "TeamID"])["PlayerID"]
    .to_dict()
)

goals_raw_df["PlayerID"] = goals_raw_df.apply(
    lambda row: player_lookup.get(
        (row["PlayerName"], row["TeamID"])
    ),
    axis=1
)

unmatched_goals = goals_raw_df[
    goals_raw_df["PlayerID"].isna()
].copy()

print("PLAYER MATCHING")
print("=" * 80)

print("Total goal records:", len(goals_raw_df))
print(
    "Matched PlayerIDs:",
    goals_raw_df["PlayerID"].notna().sum()
)
print(
    "Unmatched:",
    len(unmatched_goals)
)

if len(unmatched_goals) > 0:
    print("\nUnmatched goal scorers:")
    print(
        unmatched_goals[
            ["PlayerName", "TeamID"]
        ]
        .drop_duplicates()
        .to_string(index=False)
    )
else:
    print("\nAll goal scorers matched successfully.")

PLAYER MATCHING
Total goal records: 308
Matched PlayerIDs: 0
Unmatched: 308

Unmatched goal scorers:
              PlayerName  TeamID
         Julián Quiñones       8
            Raúl Jiménez       8
           Hwang In-Beom      31
            Oh Hyeon-Gyu      31
         Ladislav Krejcí      38
          Michal Sadílek      38
          Teboho Mokoena      24
               Luis Romo       8
            Mateo Chávez       8
          Álvaro Fidalgo       8
          Thapelo Maseko      24
              Cyle Larin      16
              Jovo Lukić      26
             Miro Muheim      44
            Breel Embolo      11
          Johan Manzambi      11
            Rubén Vargas      11
            Granit Xhaka      11
            Ermin Mahmic      26
          Jonathan David      16
           Nathan Saliba      16
           Mohamed Manai      16
           Promise David      16
       Kerim Alajbegovic      26
         Mahmoud Abunada      26
        Hassan Al-Haydos      44
        

## 18.1 Inspect Unmatched Player Names

The initial exact PlayerName + TeamID matching returned
0 matches.

This indicates that the goal source and FIFA squad dataset
use different player-name formats.

We will inspect the Players dataset for the affected teams
and compare the names manually before creating any aliases.

No fuzzy matching or automatic name replacement will be used
until the differences are understood.

In [17]:
# Inspect players from the teams appearing in the first
# unmatched goal records.

sample_unmatched = (
    unmatched_goals[
        ["PlayerName", "TeamID"]
    ]
    .drop_duplicates()
    .head(10)
)

print("UNMATCHED SOURCE NAMES")
print("=" * 80)

for _, row in sample_unmatched.iterrows():

    source_name = row["PlayerName"]
    team_id = int(row["TeamID"])

    team_name = teams_df.loc[
        teams_df["TeamID"] == team_id,
        "TeamName"
    ].iloc[0]

    print(f"\nSource: {source_name}")
    print(f"Team:   {team_name} (TeamID {team_id})")

    team_players = players_df[
        players_df["TeamID"] == team_id
    ][
        ["PlayerID", "PlayerName", "JerseyNumber"]
    ]

    print("\nPlayers.csv:")
    print(team_players.to_string(index=False))

UNMATCHED SOURCE NAMES

Source: Julián Quiñones
Team:   Mexico (TeamID 8)

Players.csv:
 PlayerID                  PlayerName  JerseyNumber
      703         Raúl RANGEL AGUILAR             1
      704       Eduardo SÁNCHEZ RAMOS             2
      705         Jasib MONTES CASTRO             3
      706      Omar ÁLVAREZ VELÁZQUEZ             4
      707       Felipe VÁSQUEZ IBARRA             5
      708                 LIRA MÉNDEZ             6
      709                 ROMO BARRÓN             7
      710           FIDALGO FERNÁNDEZ             8
      711    Alonso JIMÉNEZ RODRÍGUEZ             9
      712           Alexis VEGA ROJAS            10
      713               Tomás GIMENEZ            11
      714               ACEVEDO LÓPEZ            12
      715      Guillermo OCHOA MAGAÑA            13
      716               GONZÁLEZ ALBA            14
      717                REYES ROMERO            15
      718    Andrés QUIÑONES QUIÑONES            16
      719             PINEDA

## 18.2 Test Goal Scorer Name Matching

The goal source uses shortened player names, while Players.csv
contains FIFA squad-list names.

We will extract the last word from each source scorer name and
compare it with the last word of PlayerName in Players.csv.

The comparison will remain restricted to the same TeamID.

We will check whether this produces a unique player for every
goal event before applying the matching method.

In [18]:
def get_last_word(name):
    return str(name).strip().split()[-1]


goals_raw_df["SourceLastName"] = (
    goals_raw_df["PlayerName"]
    .apply(get_last_word)
)

players_name_check = players_df.copy()

players_name_check["PlayerLastName"] = (
    players_name_check["PlayerName"]
    .apply(get_last_word)
)

# Find candidate Players for each goal using:
# TeamID + last name

candidate_counts = []

for _, goal in goals_raw_df.iterrows():

    candidates = players_name_check[
        (players_name_check["TeamID"] == goal["TeamID"]) &
        (
            players_name_check["PlayerLastName"].str.upper()
            == goal["SourceLastName"].upper()
        )
    ]

    candidate_counts.append(len(candidates))

goals_raw_df["CandidateCount"] = candidate_counts

print("LAST-NAME MATCH TEST")
print("=" * 80)

print(
    "Total goals:",
    len(goals_raw_df)
)

print(
    "Unique candidate:",
    (goals_raw_df["CandidateCount"] == 1).sum()
)

print(
    "No candidate:",
    (goals_raw_df["CandidateCount"] == 0).sum()
)

print(
    "Multiple candidates:",
    (goals_raw_df["CandidateCount"] > 1).sum()
)

print("\nCandidate count distribution:")
print(
    goals_raw_df["CandidateCount"]
    .value_counts()
    .sort_index()
)

LAST-NAME MATCH TEST
Total goals: 308
Unique candidate: 185
No candidate: 99
Multiple candidates: 24

Candidate count distribution:
CandidateCount
0     99
1    185
2     15
3      5
4      4
Name: count, dtype: int64


## 18.3 Generate Player Name Candidates

The source uses shortened player names, while Players.csv contains
full FIFA squad-list names.

We will generate possible candidates by comparing the individual
name components within the same TeamID.

This step only generates candidates. It does not assign PlayerIDs.

The candidates will be inspected before the final matching method
is applied.

In [19]:
import re
import unicodedata


def normalize_name(text):
    text = str(text).upper().strip()

    # Remove accents for comparison
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    # Keep letters/numbers and convert everything else to spaces
    text = re.sub(r"[^A-Z0-9]+", " ", text)

    return " ".join(text.split())


players_name_check["NormalizedPlayerName"] = (
    players_name_check["PlayerName"]
    .apply(normalize_name)
)

goals_raw_df["NormalizedSourceName"] = (
    goals_raw_df["PlayerName"]
    .apply(normalize_name)
)


def get_name_tokens(name):
    return set(normalize_name(name).split())


candidate_results = []

for _, goal in goals_raw_df.iterrows():

    source_tokens = get_name_tokens(goal["PlayerName"])

    team_players = players_name_check[
        players_name_check["TeamID"] == goal["TeamID"]
    ].copy()

    candidates = []

    for _, player in team_players.iterrows():

        player_tokens = get_name_tokens(
            player["PlayerName"]
        )

        overlap = source_tokens.intersection(player_tokens)

        if overlap:
            candidates.append({
                "PlayerID": int(player["PlayerID"]),
                "PlayerName": player["PlayerName"],
                "Overlap": ", ".join(sorted(overlap)),
                "OverlapCount": len(overlap)
            })

    candidates = sorted(
        candidates,
        key=lambda x: x["OverlapCount"],
        reverse=True
    )

    candidate_results.append(candidates)


goals_raw_df["Candidates"] = candidate_results

print("NAME COMPONENT CANDIDATE TEST")
print("=" * 80)

candidate_count_distribution = (
    goals_raw_df["Candidates"]
    .apply(len)
    .value_counts()
    .sort_index()
)

print("Candidate count distribution:")
print(candidate_count_distribution)

print("\nExamples:")

for _, row in goals_raw_df.head(10).iterrows():

    print(
        f"\nSource: {row['PlayerName']} "
        f"(TeamID {row['TeamID']})"
    )

    for candidate in row["Candidates"][:5]:
        print(
            f"  -> {candidate['PlayerID']} | "
            f"{candidate['PlayerName']} | "
            f"Overlap: {candidate['Overlap']}"
        )

NAME COMPONENT CANDIDATE TEST
Candidate count distribution:
Candidates
0      20
1     230
2      32
3      10
4      13
5       2
12      1
Name: count, dtype: int64

Examples:

Source: Julián Quiñones (TeamID 8)
  -> 718 | Andrés QUIÑONES QUIÑONES | Overlap: QUINONES

Source: Raúl Jiménez (TeamID 8)
  -> 703 | Raúl RANGEL AGUILAR | Overlap: RAUL
  -> 711 | Alonso JIMÉNEZ RODRÍGUEZ | Overlap: JIMENEZ

Source: Hwang In-Beom (TeamID 31)
  -> 682 | HWANG INBEOM | Overlap: HWANG
  -> 687 | HWANG HEECHAN | Overlap: HWANG

Source: Oh Hyeon-Gyu (TeamID 31)
  -> 694 | OH HYEONGYU | Overlap: OH

Source: Ladislav Krejcí (TeamID 38)
  -> 371 | KREJČÍ KREJČÍ | Overlap: KREJCI

Source: Michal Sadílek (TeamID 38)
  -> 382 | SADÍLEK SADÍLEK | Overlap: SADILEK

Source: Teboho Mokoena (TeamID 24)
  -> 1018 | Teboho MOKOENA MOKOENA | Overlap: MOKOENA, TEBOHO

Source: Luis Romo (TeamID 8)
  -> 709 | ROMO BARRÓN | Overlap: ROMO

Source: Mateo Chávez (TeamID 8)
  -> 722 | CHÁVEZ GARCÍA | Overlap: CHAVEZ
 

## 18.4 Calculate Full-Name Similarity

The source uses shortened/display player names, while Players.csv
contains full FIFA squad-list names.

We will compare the complete normalized names within the same team
using sequence similarity.

This step generates a best candidate and similarity score for each
goal scorer.

No PlayerID will be permanently assigned yet.

In [20]:
from difflib import SequenceMatcher


def name_similarity(source_name, player_name):
    source = normalize_name(source_name)
    player = normalize_name(player_name)

    return SequenceMatcher(
        None,
        source,
        player
    ).ratio()


best_candidates = []

for _, goal in goals_raw_df.iterrows():

    team_players = players_name_check[
        players_name_check["TeamID"] == goal["TeamID"]
    ].copy()

    candidates = []

    for _, player in team_players.iterrows():

        score = name_similarity(
            goal["PlayerName"],
            player["PlayerName"]
        )

        candidates.append({
            "PlayerID": int(player["PlayerID"]),
            "PlayerName": player["PlayerName"],
            "Similarity": score
        })

    candidates = sorted(
        candidates,
        key=lambda x: x["Similarity"],
        reverse=True
    )

    best_candidates.append(candidates[:3])


goals_raw_df["BestCandidates"] = best_candidates

print("FULL-NAME SIMILARITY TEST")
print("=" * 80)

for _, row in goals_raw_df.head(15).iterrows():

    print(
        f"\nSource: {row['PlayerName']} "
        f"(TeamID {row['TeamID']})"
    )

    for candidate in row["BestCandidates"]:
        print(
            f"  -> PlayerID {candidate['PlayerID']} | "
            f"{candidate['PlayerName']} | "
            f"Similarity: {candidate['Similarity']:.3f}"
        )

FULL-NAME SIMILARITY TEST

Source: Julián Quiñones (TeamID 8)
  -> PlayerID 718 | Andrés QUIÑONES QUIÑONES | Similarity: 0.564
  -> PlayerID 703 | Raúl RANGEL AGUILAR | Similarity: 0.412
  -> PlayerID 705 | Jasib MONTES CASTRO | Similarity: 0.412

Source: Raúl Jiménez (TeamID 8)
  -> PlayerID 708 | LIRA MÉNDEZ | Similarity: 0.696
  -> PlayerID 713 | Tomás GIMENEZ | Similarity: 0.640
  -> PlayerID 711 | Alonso JIMÉNEZ RODRÍGUEZ | Similarity: 0.556

Source: Hwang In-Beom (TeamID 31)
  -> PlayerID 682 | HWANG INBEOM | Similarity: 0.960
  -> PlayerID 678 | LEE HANBEOM | Similarity: 0.583
  -> PlayerID 687 | HWANG HEECHAN | Similarity: 0.538

Source: Oh Hyeon-Gyu (TeamID 31)
  -> PlayerID 694 | OH HYEONGYU | Similarity: 0.957
  -> PlayerID 697 | JO HYEONWOO | Similarity: 0.609
  -> PlayerID 677 | KIM SEUNGGYU | Similarity: 0.500

Source: Ladislav Krejcí (TeamID 38)
  -> PlayerID 371 | KREJČÍ KREJČÍ | Similarity: 0.571
  -> PlayerID 382 | SADÍLEK SADÍLEK | Similarity: 0.400
  -> PlayerID 368

In [21]:
print("PLAYERS DATASET SAMPLE")
print("=" * 80)

print(
    players_df[
        players_df["TeamID"] == 8
    ][
        ["PlayerID", "PlayerName", "JerseyNumber"]
    ].to_string(index=False)
)

PLAYERS DATASET SAMPLE
 PlayerID                  PlayerName  JerseyNumber
      703         Raúl RANGEL AGUILAR             1
      704       Eduardo SÁNCHEZ RAMOS             2
      705         Jasib MONTES CASTRO             3
      706      Omar ÁLVAREZ VELÁZQUEZ             4
      707       Felipe VÁSQUEZ IBARRA             5
      708                 LIRA MÉNDEZ             6
      709                 ROMO BARRÓN             7
      710           FIDALGO FERNÁNDEZ             8
      711    Alonso JIMÉNEZ RODRÍGUEZ             9
      712           Alexis VEGA ROJAS            10
      713               Tomás GIMENEZ            11
      714               ACEVEDO LÓPEZ            12
      715      Guillermo OCHOA MAGAÑA            13
      716               GONZÁLEZ ALBA            14
      717                REYES ROMERO            15
      718    Andrés QUIÑONES QUIÑONES            16
      719             PINEDA ALVARADO            17
      720                Goméz VARGAS    

## 18.6 Identify Unique Goal Scorers

There are 308 goal events, but many players scored multiple goals.

We will identify the unique player names appearing in the goal
dataset.

This reduces the number of player mappings that need to be resolved
and allows us to validate each scorer against the correct team.

In [22]:
unique_scorers = (
    goals_raw_df[
        ["PlayerName", "TeamID"]
    ]
    .drop_duplicates()
    .sort_values(["TeamID", "PlayerName"])
    .reset_index(drop=True)
)

print("UNIQUE GOAL SCORERS")
print("=" * 80)

print("Total goal events:", len(goals_raw_df))
print("Unique scorers:", len(unique_scorers))

print("\nUnique scorers by team:")

scorer_counts = (
    unique_scorers
    .groupby("TeamID")
    .size()
    .reset_index(name="UniqueScorers")
)

print(scorer_counts.to_string(index=False))

UNIQUE GOAL SCORERS
Total goal events: 308
Unique scorers: 192

Unique scorers by team:
 TeamID  UniqueScorers
      1              9
      2              8
      3              5
      4              4
      5              7
      6              4
      7              7
      8              5
      9              6
     10              8
     11              5
     12              8
     13              3
     14              5
     15              4
     16              6
     17              3
     18              6
     19              6
     20              6
     21              6
     22              7
     23              6
     24              2
     25              4
     26              4
     27              3
     28              2
     29              2
     30              2
     31              2
     32              3
     33              1
     34              5
     35              5
     36              2
     37              1
     38              2
     39        

## 18.7 Generate Verified Player Candidates

We will generate up to three possible FIFA squad players for
each unique goal scorer.

Candidates are restricted to the same TeamID.

For each candidate we will display:

- Source player name
- TeamID
- Candidate PlayerID
- FIFA squad PlayerName
- JerseyNumber
- Name overlap

This will help us identify the correct mapping before assigning
PlayerIDs to the 308 goal events.

In [23]:
# Build candidates for each unique scorer

unique_scorer_candidates = []

for _, scorer in unique_scorers.iterrows():

    source_name = scorer["PlayerName"]
    team_id = int(scorer["TeamID"])

    source_tokens = get_name_tokens(source_name)

    team_players = players_name_check[
        players_name_check["TeamID"] == team_id
    ].copy()

    candidates = []

    for _, player in team_players.iterrows():

        player_tokens = get_name_tokens(
            player["PlayerName"]
        )

        overlap = source_tokens.intersection(player_tokens)

        # Require at least one matching name component
        if overlap:

            score = name_similarity(
                source_name,
                player["PlayerName"]
            )

            candidates.append({
                "PlayerID": int(player["PlayerID"]),
                "PlayerName": player["PlayerName"],
                "JerseyNumber": int(player["JerseyNumber"]),
                "Similarity": score,
                "Overlap": ", ".join(sorted(overlap))
            })

    candidates = sorted(
        candidates,
        key=lambda x: (
            x["Similarity"],
            len(x["Overlap"])
        ),
        reverse=True
    )

    unique_scorer_candidates.append({
        "PlayerName": source_name,
        "TeamID": team_id,
        "Candidates": candidates[:3]
    })


# Display the first 20 unique scorers and their candidates

print("PLAYER CANDIDATE REVIEW")
print("=" * 100)

for item in unique_scorer_candidates[:20]:

    team_name = teams_df.loc[
        teams_df["TeamID"] == item["TeamID"],
        "TeamName"
    ].iloc[0]

    print(
        f"\nSource: {item['PlayerName']} | "
        f"Team: {team_name} | "
        f"TeamID: {item['TeamID']}"
    )

    if not item["Candidates"]:
        print("  -> NO CANDIDATES")
    else:
        for candidate in item["Candidates"]:
            print(
                f"  -> PlayerID {candidate['PlayerID']} | "
                f"{candidate['PlayerName']} | "
                f"Jersey #{candidate['JerseyNumber']} | "
                f"Similarity {candidate['Similarity']:.3f} | "
                f"Overlap: {candidate['Overlap']}"
            )

PLAYER CANDIDATE REVIEW

Source: Alexis Mac Allister | Team: Argentina | TeamID: 1
  -> PlayerID 46 | MAC ALLISTER | Jersey #20 | Similarity 0.774 | Overlap: ALLISTER, MAC

Source: Cristian Romero | Team: Argentina | TeamID: 1
  -> PlayerID 39 | Gabriel ROMERO | Jersey #13 | Similarity 0.621 | Overlap: ROMERO

Source: Diney Borges | Team: Argentina | TeamID: 1
  -> NO CANDIDATES

Source: Enzo Fernández | Team: Argentina | TeamID: 1
  -> PlayerID 50 | Jeremías FERNÁNDEZ | Jersey #24 | Similarity 0.688 | Overlap: FERNANDEZ

Source: Giovani Lo Celso | Team: Argentina | TeamID: 1
  -> PlayerID 37 | LO CELSO | Jersey #11 | Similarity 0.667 | Overlap: CELSO, LO

Source: Julián Álvarez | Team: Argentina | TeamID: 1
  -> PlayerID 35 | ÁLVAREZ | Jersey #9 | Similarity 0.667 | Overlap: ALVAREZ

Source: Lautaro Martínez | Team: Argentina | TeamID: 1
  -> PlayerID 49 | Emiliano MARTÍNEZ | Jersey #23 | Similarity 0.727 | Overlap: MARTINEZ
  -> PlayerID 44 | PAZ MARTÍNEZ | Jersey #18 | Similarity 0.

In [24]:
# Analyze the strongest candidate for every unique scorer

review_rows = []

for item in unique_scorer_candidates:

    candidates = item["Candidates"]

    if not candidates:
        status = "NO CANDIDATE"
        best_score = None
        second_score = None

    else:
        best_score = candidates[0]["Similarity"]

        if len(candidates) > 1:
            second_score = candidates[1]["Similarity"]
        else:
            second_score = 0

        # Only classify as clearly strong if the best candidate
        # is substantially better than the second candidate.
        if best_score >= 0.85 and (best_score - second_score) >= 0.10:
            status = "STRONG"
        elif best_score >= 0.70 and (best_score - second_score) >= 0.10:
            status = "LIKELY"
        else:
            status = "AMBIGUOUS"

    review_rows.append({
        "PlayerName": item["PlayerName"],
        "TeamID": item["TeamID"],
        "Status": status,
        "BestScore": best_score,
        "SecondScore": second_score
    })

scorer_review_df = pd.DataFrame(review_rows)

print("SCORER MATCHING REVIEW")
print("=" * 100)

print(
    scorer_review_df["Status"]
    .value_counts()
)

print("\nTotal unique scorers:", len(scorer_review_df))
print(
    "Strong:",
    (scorer_review_df["Status"] == "STRONG").sum()
)
print(
    "Likely:",
    (scorer_review_df["Status"] == "LIKELY").sum()
)
print(
    "Ambiguous:",
    (scorer_review_df["Status"] == "AMBIGUOUS").sum()
)
print(
    "No candidate:",
    (scorer_review_df["Status"] == "NO CANDIDATE").sum()
)

SCORER MATCHING REVIEW
Status
AMBIGUOUS       105
LIKELY           45
STRONG           23
NO CANDIDATE     19
Name: count, dtype: int64

Total unique scorers: 192
Strong: 23
Likely: 45
Ambiguous: 105
No candidate: 19


In [25]:
print("AVAILABLE PLAYER DATA")
print("=" * 100)

print("players_df columns:")
print(players_df.columns.tolist())

print("\nVariables related to squad/player data:")
print([
    name for name in globals()
    if any(
        word in name.lower()
        for word in [
            "player",
            "squad",
            "pdf",
            "team"
        ]
    )
])

AVAILABLE PLAYER DATA
players_df columns:
['PlayerID', 'PlayerName', 'TeamID', 'Position', 'JerseyNumber']

Variables related to squad/player data:
['teams_df', 'players_df', 'team_name_to_id', 'team_name', 'team_id', 'source_team_names', 'database_team_names', 'team_aliases', 'team_ids', 'team1_normalized', 'team2_normalized', 'team1_id', 'team2_id', 'player_lookup', 'team_players', 'players_name_check', 'player', 'player_tokens']


## 18.10 Inspect Complete Goal Record Structure

The goal records currently expose the player name and goal minute.

Before attempting any name matching, we will inspect the complete
raw goal objects to determine whether the source contains an
additional player identifier, jersey number, ID, or other field
that can be used for reliable player matching.

In [27]:
import requests

print("COMPLETE GOAL RECORD INSPECTION")
print("=" * 100)

response = requests.get(
    "https://foot-boll.com/api/data.php?action=match&id=0",
    timeout=30
)

response.raise_for_status()

match_data = response.json()["match"]

examples_found = 0

for team_key in ["goals1", "goals2"]:

    goals = match_data.get(team_key, [])

    for goal in goals:

        print("\nTeam section:", team_key)
        print("Goal object:")
        print(goal)

        examples_found += 1

        if examples_found >= 10:
            break

    if examples_found >= 10:
        break

COMPLETE GOAL RECORD INSPECTION

Team section: goals1
Goal object:
{'name': 'Julián Quiñones', 'minute': '9'}

Team section: goals1
Goal object:
{'name': 'Raúl Jiménez', 'minute': '67'}


## 18.11 Inspect Scorers Endpoint

The match goal records contain only the scorer's display name and
goal minute.

We will inspect the dedicated scorers endpoint to determine whether
it provides additional information such as jersey number, player ID,
team, or another reliable identifier.

No player matching will be performed until the available fields
are verified.

In [28]:
import requests

print("SCORERS ENDPOINT INSPECTION")
print("=" * 100)

scorers_url = "https://foot-boll.com/api/data.php?action=scorers"

response = requests.get(
    scorers_url,
    timeout=30
)

print("HTTP Status:", response.status_code)
print("Response Size:", len(response.content), "bytes")

response.raise_for_status()

scorers_data = response.json()

print("\nTop-level type:", type(scorers_data))

if isinstance(scorers_data, dict):
    print("Top-level keys:")
    print(list(scorers_data.keys()))

    for key, value in scorers_data.items():
        if isinstance(value, list):
            print(f"\nList key: {key}")
            print("Number of records:", len(value))

            if len(value) > 0:
                print("\nFirst 5 records:")
                for record in value[:5]:
                    print(record)

            break

SCORERS ENDPOINT INSPECTION
HTTP Status: 200
Response Size: 20204 bytes

Top-level type: <class 'dict'>
Top-level keys:
['ok', 'action', 'updated', 'scorers']

List key: scorers
Number of records: 178

First 5 records:
{'name': 'Kylian Mbappé', 'team': 'France', 'goals': 10}
{'name': 'Lionel Messi', 'team': 'Argentina', 'goals': 8}
{'name': 'Erling Haaland', 'team': 'Norway', 'goals': 7}
{'name': 'Jude Bellingham', 'team': 'England', 'goals': 7}
{'name': 'Harry Kane', 'team': 'England', 'goals': 6}


## 18.12 Compare Goal Scorers with Physical Player Data

The physical player endpoint contains player display names, teams,
and jersey numbers.

Because the Players table has a unique TeamID + JerseyNumber
combination, we will test whether the goal scorer names can be
matched to those physical records.

This is only a matching test. No PlayerIDs will be assigned yet.

In [29]:
import requests
import pandas as pd

print("SCORER → PHYSICAL PLAYER MATCHING TEST")
print("=" * 100)

# Get scorer records
scorers_url = "https://foot-boll.com/api/data.php?action=scorers"

scorers_response = requests.get(
    scorers_url,
    timeout=30
)

scorers_response.raise_for_status()

scorers_records = scorers_response.json()["scorers"]

scorers_df = pd.DataFrame(scorers_records)

print("Scorer records:", len(scorers_df))
print("Scorer columns:", scorers_df.columns.tolist())

# Get physical player records
physical_url = "https://foot-boll.com/api/data.php?action=physical"

physical_response = requests.get(
    physical_url,
    timeout=30
)

physical_response.raise_for_status()

physical_records = physical_response.json()["players"]

physical_df = pd.DataFrame(physical_records)

print("\nPhysical player records:", len(physical_df))
print("Physical columns:", physical_df.columns.tolist())

print("\nFirst 10 physical player records:")
print(
    physical_df[["name", "team", "num"]]
    .head(10)
    .to_string(index=False)
)

# Normalize names for exact comparison
def normalize_name(name):
    return (
        str(name)
        .upper()
        .replace("-", "")
        .replace("'", "")
        .replace(" ", "")
    )

scorers_df["NameNormalized"] = scorers_df["name"].apply(normalize_name)
physical_df["NameNormalized"] = physical_df["name"].apply(normalize_name)

# Team aliases already verified in Notebook 06
team_aliases = {
    "Bosnia & Herzegovina": "Bosnia and Herzegovina",
    "Cape Verde": "Cabo Verde",
    "Czech Republic": "Czechia",
    "DR Congo": "Congo DR",
    "Iran": "IR Iran",
    "Ivory Coast": "Côte d'Ivoire",
    "South Korea": "Korea Republic",
    "Turkey": "Türkiye"
}

scorers_df["TeamForMatching"] = (
    scorers_df["team"]
    .replace(team_aliases)
)

# Exact name + team matching
physical_lookup = {
    (row["NameNormalized"], row["team"]): row["num"]
    for _, row in physical_df.iterrows()
}

scorers_df["PhysicalJersey"] = scorers_df.apply(
    lambda row: physical_lookup.get(
        (row["NameNormalized"], row["TeamForMatching"])
    ),
    axis=1
)

matched = scorers_df["PhysicalJersey"].notna().sum()
unmatched = scorers_df["PhysicalJersey"].isna().sum()

print("\nEXACT MATCH RESULTS")
print("-" * 100)
print("Total scorer records:", len(scorers_df))
print("Exact name + team matches:", matched)
print("Unmatched:", unmatched)

print("\nFirst 20 matched scorers:")
print(
    scorers_df[
        scorers_df["PhysicalJersey"].notna()
    ][["name", "team", "goals", "PhysicalJersey"]]
    .head(20)
    .to_string(index=False)
)

print("\nFirst 20 unmatched scorers:")
print(
    scorers_df[
        scorers_df["PhysicalJersey"].isna()
    ][["name", "team", "goals"]]
    .head(20)
    .to_string(index=False)
)

SCORER → PHYSICAL PLAYER MATCHING TEST
Scorer records: 178
Scorer columns: ['name', 'team', 'goals']

Physical player records: 1029
Physical columns: ['name', 'team', 'teamAr', 'flag', 'photo', 'num', 'm', 'dist', 'sprints', 'hsr', 'top', 'zones', 'lb', 'lbv', 'cross']

First 10 physical player records:
               name        team  num
              RODRI       Spain   16
Alexis MAC ALLISTER   Argentina   20
     Marc CUCURELLA       Spain   24
        Pau CUBARSI       Spain   22
      Michael OLISE      France   11
     Enzo FERNANDEZ   Argentina   24
    Aymeric LAPORTE       Spain   14
    Elliot ANDERSON     England    8
         Harry KANE     England    9
       Remo FREULER Switzerland    8

EXACT MATCH RESULTS
----------------------------------------------------------------------------------------------------
Total scorer records: 178
Exact name + team matches: 125
Unmatched: 53

First 20 matched scorers:
                name        team  goals  PhysicalJersey
        Lion

## 18.13 Inspect Unmatched Scorers

The exact name + team comparison matched 125 of 178 scorer records.

We will now inspect all unmatched scorer records together with the
players available for the same team in the physical dataset.

The goal is to identify whether the remaining differences are caused
by display-name formatting, accents, abbreviations, or other naming
differences.

No PlayerID assignments will be made yet.

In [30]:
# Inspect every unmatched scorer and the physical players from the same team

unmatched_scorers = scorers_df[
    scorers_df["PhysicalJersey"].isna()
].copy()

print("UNMATCHED SCORERS")
print("=" * 100)

print("Number of unmatched scorer records:", len(unmatched_scorers))

for _, scorer in unmatched_scorers.iterrows():

    team = scorer["TeamForMatching"]

    team_players = physical_df[
        physical_df["team"] == team
    ][["name", "num"]].copy()

    print("\n" + "-" * 100)
    print(
        f"Scorer: {scorer['name']} | "
        f"Team: {scorer['team']} | "
        f"Goals: {scorer['goals']}"
    )

    print("Physical players for this team:")

    if len(team_players) == 0:
        print("  NO TEAM FOUND")
    else:
        print(
            team_players
            .sort_values("num")
            .to_string(index=False)
        )

UNMATCHED SCORERS
Number of unmatched scorer records: 53

----------------------------------------------------------------------------------------------------
Scorer: Kylian Mbappé | Team: France | Goals: 10
Physical players for this team:
                name  num
          Malo GUSTO    2
         Lucas DIGNE    3
     Dayot UPAMECANO    4
        Jules KOUNDE    5
           Manu KONE    6
     Ousmane DEMBELE    7
 Aurelien TCHOUAMENI    8
       Marcus THURAM    9
       Kylian MBAPPE   10
       Michael OLISE   11
     Bradley BARCOLA   12
       Adrien RABIOT   14
     Ibrahima KONATE   15
        Mike MAIGNAN   16
      William SALIBA   17
  Warren ZAIRE-EMERY   18
      Theo HERNANDEZ   19
         Desire DOUE   20
Jean-Philippe MATETA   22
        Rayan CHERKI   24
   Maghnes AKLIOUCHE   25
     Maxence LACROIX   26

----------------------------------------------------------------------------------------------------
Scorer: Ousmane Dembélé | Team: France | Goals: 6
Physical p

## 18.14 Normalize Scorer and Physical Player Names

The previous exact comparison was case-sensitive and accent-sensitive.

We will normalize names by:

- converting to uppercase
- removing accents
- removing spaces
- removing punctuation

The team will remain a required matching condition.

This is a matching test only. No PlayerIDs will be assigned yet.

In [31]:
import unicodedata
import re

def normalize_player_name(name):
    """
    Normalize a player name for reliable text comparison.
    """
    text = str(name)

    # Remove accents/diacritics
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        char for char in text
        if not unicodedata.combining(char)
    )

    # Uppercase
    text = text.upper()

    # Keep letters and numbers only
    text = re.sub(r"[^A-Z0-9]", "", text)

    return text


# Create normalized names
scorers_df["NameNormalized2"] = scorers_df["name"].apply(
    normalize_player_name
)

physical_df["NameNormalized2"] = physical_df["name"].apply(
    normalize_player_name
)

# Build lookup using normalized name + normalized team
physical_lookup_normalized = {}

for _, row in physical_df.iterrows():

    key = (
        row["NameNormalized2"],
        normalize_player_name(row["team"])
    )

    physical_lookup_normalized[key] = row["num"]


# Test normalized matching
scorers_df["PhysicalJerseyNormalized"] = scorers_df.apply(
    lambda row: physical_lookup_normalized.get(
        (
            row["NameNormalized2"],
            normalize_player_name(row["TeamForMatching"])
        )
    ),
    axis=1
)


matched_normalized = (
    scorers_df["PhysicalJerseyNormalized"].notna().sum()
)

unmatched_normalized = (
    scorers_df["PhysicalJerseyNormalized"].isna().sum()
)

print("NORMALIZED MATCH RESULTS")
print("=" * 100)

print("Total scorer records:", len(scorers_df))
print("Matched:", matched_normalized)
print("Unmatched:", unmatched_normalized)

print("\nNewly matched records:")
print("-" * 100)

newly_matched = scorers_df[
    scorers_df["PhysicalJersey"].isna()
    & scorers_df["PhysicalJerseyNormalized"].notna()
]

print(
    newly_matched[
        [
            "name",
            "team",
            "goals",
            "PhysicalJerseyNormalized"
        ]
    ].to_string(index=False)
)

print("\nStill unmatched:")
print("-" * 100)

still_unmatched = scorers_df[
    scorers_df["PhysicalJerseyNormalized"].isna()
]

print(
    still_unmatched[
        ["name", "team", "goals"]
    ].to_string(index=False)
)

NORMALIZED MATCH RESULTS
Total scorer records: 178
Matched: 152
Unmatched: 26

Newly matched records:
----------------------------------------------------------------------------------------------------
             name        team  goals  PhysicalJerseyNormalized
    Kylian Mbappé      France     10                      10.0
  Ousmane Dembélé      France      6                       7.0
     Ismaïla Sarr     Senegal      4                      18.0
  Julián Quiñones      Mexico      4                      16.0
  Vinícius Júnior      Brazil      4                       7.0
 Lautaro Martínez   Argentina      3                      22.0
     Raúl Jiménez      Mexico      3                       9.0
     Daniel Muñoz    Colombia      2                       2.0
   Enzo Fernández   Argentina      2                      24.0
      Maxi Araújo     Uruguay      2                      20.0
     Rubén Vargas Switzerland      2                      17.0
      Désiré Doué      France      1     

## 18.15 Inspect Remaining Unmatched Scorers

After normalization, 152 of 178 scorer records matched the physical
player dataset.

There are 26 remaining unmatched records.

We will inspect each unmatched scorer against the physical players
from the same team and compare the name components.

The team remains a hard constraint, and no PlayerID assignment will
be made until the remaining matches are verified.

In [32]:
print("REMAINING UNMATCHED SCORERS")
print("=" * 100)

still_unmatched = scorers_df[
    scorers_df["PhysicalJerseyNormalized"].isna()
].copy()

for _, scorer in still_unmatched.iterrows():

    team = scorer["TeamForMatching"]

    scorer_tokens = set(
        re.findall(
            r"[A-Z0-9]+",
            scorer["NameNormalized2"]
        )
    )

    team_players = physical_df[
        physical_df["team"] == team
    ][["name", "num", "NameNormalized2"]].copy()

    print("\n" + "-" * 100)
    print(
        f"Scorer: {scorer['name']} | "
        f"Team: {scorer['team']} | "
        f"Goals: {scorer['goals']}"
    )

    if len(team_players) == 0:
        print("NO PHYSICAL PLAYERS FOUND FOR THIS TEAM")
        continue

    candidates = []

    for _, player in team_players.iterrows():

        player_tokens = set(
            re.findall(
                r"[A-Z0-9]+",
                player["NameNormalized2"]
            )
        )

        # Compare character-normalized names
        scorer_name = scorer["NameNormalized2"]
        player_name = player["NameNormalized2"]

        # Component containment / overlap
        common_chars = set(scorer_name) & set(player_name)

        candidates.append({
            "PhysicalName": player["name"],
            "Jersey": int(player["num"]),
            "CommonCharacters": len(common_chars),
            "PlayerNameNormalized": player_name
        })

    candidates_df = (
        pd.DataFrame(candidates)
        .sort_values(
            "CommonCharacters",
            ascending=False
        )
        .head(8)
    )

    print(
        candidates_df[
            ["PhysicalName", "Jersey", "CommonCharacters"]
        ].to_string(index=False)
    )

REMAINING UNMATCHED SCORERS

----------------------------------------------------------------------------------------------------
Scorer: Yoane Wissa | Team: DR Congo | Goals: 3
NO PHYSICAL PLAYERS FOUND FOR THIS TEAM

----------------------------------------------------------------------------------------------------
Scorer: Amad Diallo | Team: Ivory Coast | Goals: 2
NO PHYSICAL PLAYERS FOUND FOR THIS TEAM

----------------------------------------------------------------------------------------------------
Scorer: Ermin Mahmic | Team: Bosnia & Herzegovina | Goals: 2
NO PHYSICAL PLAYERS FOUND FOR THIS TEAM

----------------------------------------------------------------------------------------------------
Scorer: Nicolas Pépé | Team: Ivory Coast | Goals: 2
NO PHYSICAL PLAYERS FOUND FOR THIS TEAM

----------------------------------------------------------------------------------------------------
Scorer: Ramin Rezaeian | Team: Iran | Goals: 2
NO PHYSICAL PLAYERS FOUND FOR THIS TEAM

--

## 18.16 Normalize Team Names Across Scorer and Physical Data

Some unmatched scorers belong to teams whose names differ between
the scorer endpoint and physical endpoint.

We will apply the previously verified FIFA team aliases to both
datasets so that the same team is represented consistently.

After this correction, we will repeat the exact normalized
name + team matching.

No PlayerIDs will be assigned yet.

In [33]:
# Apply the verified team aliases consistently to both datasets

physical_df["TeamForMatching"] = (
    physical_df["team"]
    .replace(team_aliases)
)

# Rebuild the normalized lookup using the corrected team names
physical_lookup_normalized = {}

for _, row in physical_df.iterrows():

    key = (
        row["NameNormalized2"],
        normalize_player_name(row["TeamForMatching"])
    )

    physical_lookup_normalized[key] = row["num"]


# Repeat normalized name + team matching
scorers_df["PhysicalJerseyNormalized"] = scorers_df.apply(
    lambda row: physical_lookup_normalized.get(
        (
            row["NameNormalized2"],
            normalize_player_name(row["TeamForMatching"])
        )
    ),
    axis=1
)


matched_normalized = (
    scorers_df["PhysicalJerseyNormalized"].notna().sum()
)

unmatched_normalized = (
    scorers_df["PhysicalJerseyNormalized"].isna().sum()
)

print("CORRECTED NORMALIZED MATCH RESULTS")
print("=" * 100)

print("Total scorer records:", len(scorers_df))
print("Matched:", matched_normalized)
print("Unmatched:", unmatched_normalized)

print("\nNewly matched records:")
print("-" * 100)

newly_matched = scorers_df[
    scorers_df["PhysicalJerseyNormalized"].notna()
    & scorers_df["PhysicalJersey"].isna()
]

print(
    newly_matched[
        [
            "name",
            "team",
            "goals",
            "PhysicalJerseyNormalized"
        ]
    ].to_string(index=False)
)

print("\nStill unmatched:")
print("-" * 100)

still_unmatched = scorers_df[
    scorers_df["PhysicalJerseyNormalized"].isna()
]

print(
    still_unmatched[
        ["name", "team", "goals"]
    ].to_string(index=False)
)

CORRECTED NORMALIZED MATCH RESULTS
Total scorer records: 178
Matched: 173
Unmatched: 5

Newly matched records:
----------------------------------------------------------------------------------------------------
              name                 team  goals  PhysicalJerseyNormalized
     Kylian Mbappé               France     10                      10.0
   Ousmane Dembélé               France      6                       7.0
      Ismaïla Sarr              Senegal      4                      18.0
   Julián Quiñones               Mexico      4                      16.0
   Vinícius Júnior               Brazil      4                       7.0
  Lautaro Martínez            Argentina      3                      22.0
      Raúl Jiménez               Mexico      3                       9.0
       Yoane Wissa             DR Congo      3                      20.0
       Amad Diallo          Ivory Coast      2                      15.0
      Daniel Muñoz             Colombia      2            

## 18.17 Inspect Final Unmatched Scorers

After correcting the team-name aliases and normalizing player names,
173 of 178 scorer records now match the physical player dataset.

Only 5 scorer records remain unmatched.

We will inspect these five against the physical players from their
respective teams to determine the exact source of the naming
difference.

No PlayerID assignments will be made until these five are verified.

In [35]:
print("FINAL 5 UNMATCHED SCORERS")
print("=" * 100)

final_unmatched = scorers_df[
    scorers_df["PhysicalJerseyNormalized"].isna()
].copy()

print("Number of unmatched:", len(final_unmatched))

for _, scorer in final_unmatched.iterrows():

    team = scorer["TeamForMatching"]

    print("\n" + "-" * 100)
    print(
        f"Scorer: {scorer['name']} | "
        f"Source Team: {scorer['team']} | "
        f"Goals: {scorer['goals']}"
    )

    team_players = physical_df[
        physical_df["TeamForMatching"] == team
    ][["name", "num"]].copy()

    print("\nPhysical players for this team:")

    if len(team_players) == 0:
        print("NO PLAYERS FOUND")
    else:
        print(
            team_players
            .sort_values("num")
            .to_string(index=False)
        )

FINAL 5 UNMATCHED SCORERS
Number of unmatched: 5

----------------------------------------------------------------------------------------------------
Scorer: Agustín Cano | Source Team: Uruguay | Goals: 1

Physical players for this team:
                name  num
       Sergio ROCHET    1
   Sebastian CACERES    3
       Manuel UGARTE    5
   Rodrigo BENTANCUR    6
  Nicolas DE LA CRUZ    7
   Federico VALVERDE    8
        Darwin NUNEZ    9
    Guillermo VARELA   13
    Agustin CANOBBIO   14
     Mathias OLIVERA   16
         Matias VINA   17
     Brian RODRIGUEZ   18
     Rodrigo AGUIRRE   19
         Maxi ARAUJO   20
      Federico VINAS   21
    Fernando MUSLERA   23
Juan Manuel SANABRIA   25

----------------------------------------------------------------------------------------------------
Scorer: Kaan Ayhan | Source Team: Turkey | Goals: 1

Physical players for this team:
               name  num
         Zeki CELIK    2
      Merih DEMIRAL    3
        Salih OZCAN    5
      

## 18.18 Identify Candidates for the Final Five Scorers

The final five scorer records are still unmatched.

Instead of printing every player on each team, we will calculate
name similarity against players from the same team and display only
the best candidates.

This is an investigation step only. We will not assign PlayerIDs
based solely on similarity.

In [36]:
from difflib import SequenceMatcher

print("FINAL 5 SCORER CANDIDATES")
print("=" * 100)

for _, scorer in final_unmatched.iterrows():

    team = scorer["TeamForMatching"]
    scorer_name = scorer["name"]

    team_players = physical_df[
        physical_df["TeamForMatching"] == team
    ][["name", "num"]].copy()

    candidates = []

    for _, player in team_players.iterrows():

        scorer_norm = normalize_player_name(scorer_name)
        player_norm = normalize_player_name(player["name"])

        similarity = SequenceMatcher(
            None,
            scorer_norm,
            player_norm
        ).ratio()

        candidates.append({
            "Scorer": scorer_name,
            "Team": scorer["team"],
            "Goals": scorer["goals"],
            "PhysicalName": player["name"],
            "Jersey": int(player["num"]),
            "Similarity": round(similarity, 3)
        })

    candidates_df = pd.DataFrame(candidates)

    print("\n" + "-" * 100)

    if len(candidates_df) == 0:
        print(
            f"{scorer_name} | {scorer['team']} | "
            "NO PHYSICAL PLAYERS"
        )
    else:
        print(
            candidates_df
            .sort_values("Similarity", ascending=False)
            .head(5)
            .to_string(index=False)
        )

FINAL 5 SCORER CANDIDATES

----------------------------------------------------------------------------------------------------
      Scorer    Team  Goals      PhysicalName  Jersey  Similarity
Agustín Cano Uruguay      1  Agustin CANOBBIO      14       0.846
Agustín Cano Uruguay      1 Sebastian CACERES       3       0.519
Agustín Cano Uruguay      1       Matias VINA      17       0.476
Agustín Cano Uruguay      1      Darwin NUNEZ       9       0.364
Agustín Cano Uruguay      1   Mathias OLIVERA      16       0.320

----------------------------------------------------------------------------------------------------
    Scorer   Team  Goals     PhysicalName  Jersey  Similarity
Kaan Ayhan Turkey      1 Hakan CALHANOGLU      10       0.500
Kaan Ayhan Turkey      1     Kenan YILDIZ      11       0.400
Kaan Ayhan Turkey      1         Can UZUN      26       0.375
Kaan Ayhan Turkey      1      Salih OZCAN       5       0.316
Kaan Ayhan Turkey      1    Ugurcan CAKIR      23       0.286

-

## 18.19 Compact Review of Final Five Scorer Candidates

The similarity test identified possible candidates, but similarity
alone is not enough to create a PlayerID mapping.

We will display the final five scorers and their top candidates in a
compact format so that every case can be reviewed completely.

In [37]:
print("FINAL FIVE — COMPLETE CANDIDATE REVIEW")
print("=" * 100)

for _, scorer in final_unmatched.iterrows():

    team = scorer["TeamForMatching"]
    scorer_name = scorer["name"]

    team_players = physical_df[
        physical_df["TeamForMatching"] == team
    ][["name", "num"]].copy()

    candidates = []

    for _, player in team_players.iterrows():

        similarity = SequenceMatcher(
            None,
            normalize_player_name(scorer_name),
            normalize_player_name(player["name"])
        ).ratio()

        candidates.append({
            "name": player["name"],
            "jersey": int(player["num"]),
            "similarity": round(similarity, 3)
        })

    candidates_df = (
        pd.DataFrame(candidates)
        .sort_values("similarity", ascending=False)
        .head(5)
    )

    print(
        f"\n{scorer_name} | "
        f"{scorer['team']} | "
        f"Goals: {scorer['goals']}"
    )

    for _, candidate in candidates_df.iterrows():
        print(
            f"  → {candidate['name']} "
            f"(#{candidate['jersey']}, "
            f"similarity={candidate['similarity']})"
        )

FINAL FIVE — COMPLETE CANDIDATE REVIEW

Agustín Cano | Uruguay | Goals: 1
  → Agustin CANOBBIO (#14, similarity=0.846)
  → Sebastian CACERES (#3, similarity=0.519)
  → Matias VINA (#17, similarity=0.476)
  → Darwin NUNEZ (#9, similarity=0.364)
  → Mathias OLIVERA (#16, similarity=0.32)

Kaan Ayhan | Turkey | Goals: 1
  → Hakan CALHANOGLU (#10, similarity=0.5)
  → Kenan YILDIZ (#11, similarity=0.4)
  → Can UZUN (#26, similarity=0.375)
  → Salih OZCAN (#5, similarity=0.316)
  → Ugurcan CAKIR (#23, similarity=0.286)

Leo Østigard | Norway | Goals: 1
  → Leo OSTIGARD (#4, similarity=0.952)
  → Thelo AASGAARD (#19, similarity=0.609)
  → Martin ODEGAARD (#10, similarity=0.5)
  → Erling HAALAND (#9, similarity=0.435)
  → Kristian THORSTVEDT (#18, similarity=0.429)

Neymar | Brazil | Goals: 1
  → NEYMAR JR (#10, similarity=0.857)
  → CASEMIRO (#5, similarity=0.429)
  → BRUNO GUIMARAES (#8, similarity=0.4)
  → MARQUINHOS (#4, similarity=0.375)
  → GABRIEL MARTINELLI (#22, similarity=0.348)

Ste

## 18.20 Verify Final Five Against the Official Players Table

The physical dataset does not contain every squad player, so the final
five scorer records will now be checked directly against Players.csv.

We will search within the scorer's TeamID and display players whose
names share meaningful name components with the scorer.

This is the final verification step before creating the PlayerID
mapping.

In [38]:
print("FINAL FIVE — OFFICIAL PLAYERS TABLE REVIEW")
print("=" * 100)

for _, scorer in final_unmatched.iterrows():

    source_team = scorer["team"]
    database_team = scorer["TeamForMatching"]

    # Get TeamID
    team_id = team_name_to_id[database_team]

    # Players from this team
    team_players = players_df[
        players_df["TeamID"] == team_id
    ][["PlayerID", "PlayerName", "Position", "JerseyNumber"]].copy()

    scorer_tokens = set(
        re.findall(
            r"[A-Z0-9]+",
            normalize_player_name(scorer["name"])
        )
    )

    candidates = []

    for _, player in team_players.iterrows():

        player_tokens = set(
            re.findall(
                r"[A-Z0-9]+",
                normalize_player_name(player["PlayerName"])
            )
        )

        overlap = scorer_tokens.intersection(player_tokens)

        similarity = SequenceMatcher(
            None,
            normalize_player_name(scorer["name"]),
            normalize_player_name(player["PlayerName"])
        ).ratio()

        candidates.append({
            "PlayerID": int(player["PlayerID"]),
            "PlayerName": player["PlayerName"],
            "Jersey": int(player["JerseyNumber"]),
            "Overlap": ", ".join(sorted(overlap)) if overlap else "-",
            "Similarity": round(similarity, 3)
        })

    candidates_df = (
        pd.DataFrame(candidates)
        .sort_values(
            ["Similarity", "PlayerID"],
            ascending=[False, True]
        )
        .head(8)
    )

    print(
        f"\n{scorer['name']} | "
        f"{source_team} | "
        f"TeamID: {team_id} | "
        f"Goals: {scorer['goals']}"
    )

    for _, candidate in candidates_df.iterrows():

        print(
            f"  → PlayerID {candidate['PlayerID']} | "
            f"{candidate['PlayerName']} | "
            f"#{candidate['Jersey']} | "
            f"Overlap: {candidate['Overlap']} | "
            f"Similarity: {candidate['Similarity']}"
        )

FINAL FIVE — OFFICIAL PLAYERS TABLE REVIEW

Agustín Cano | Uruguay | TeamID: 36 | Goals: 1
  → PlayerID 1185 | MARTÍNEZ TORANZA | #15 | Overlap: - | Similarity: 0.462
  → PlayerID 1191 | Sebastian VIÑAS BARBOZA | #21 | Overlap: - | Similarity: 0.438
  → PlayerID 1184 | CANOBBIO | #14 | Overlap: - | Similarity: 0.421
  → PlayerID 1182 | Andres MELE CASTAÑERO | #12 | Overlap: - | Similarity: 0.4
  → PlayerID 1189 | Sebastian AGUIRRE SOTO | #19 | Overlap: - | Similarity: 0.387
  → PlayerID 1193 | Fernando MUSLERA MICOL | #23 | Overlap: - | Similarity: 0.387
  → PlayerID 1176 | BENTANCUR COLMÁN | #6 | Overlap: - | Similarity: 0.385
  → PlayerID 1175 | UGARTE RIBEIRO | #5 | Overlap: - | Similarity: 0.333

Kaan Ayhan | Turkey | TeamID: 47 | Goals: 1
  → PlayerID 1166 | Kaan AYHAN KAAN | #22 | Overlap: - | Similarity: 0.818
  → PlayerID 1149 | Salih ÖZCAN ÖZCAN | #5 | Overlap: - | Similarity: 0.417
  → PlayerID 1156 | Altay BAYINDIR ALTAY | #12 | Overlap: - | Similarity: 0.37
  → PlayerID 116

## 18.21 Verify the Remaining Unresolved Scorers

Kaan Ayhan and Leo Østigard have reliable matches in Players.csv.

Three scorer records still require verification:

- Agustín Cano — Uruguay
- Neymar — Brazil
- Stephan Eustáquio — Canada

We will display the complete squad for only these three teams so that
the exact official player records can be identified without relying
on truncated output or unsupported fuzzy matching.

In [39]:
unresolved_names = [
    "Agustín Cano",
    "Neymar",
    "Stephan Eustáquio"
]

print("UNRESOLVED SCORERS — OFFICIAL PLAYERS")
print("=" * 100)

for scorer_name in unresolved_names:

    scorer_row = final_unmatched[
        final_unmatched["name"] == scorer_name
    ].iloc[0]

    team_name = scorer_row["TeamForMatching"]
    team_id = team_name_to_id[team_name]

    team_players = players_df[
        players_df["TeamID"] == team_id
    ][
        ["PlayerID", "PlayerName", "Position", "JerseyNumber"]
    ].sort_values("JerseyNumber")

    print(
        f"\n{scorer_name} | "
        f"Team: {scorer_row['team']} | "
        f"TeamID: {team_id}"
    )

    print(
        team_players.to_string(index=False)
    )

UNRESOLVED SCORERS — OFFICIAL PLAYERS

Agustín Cano | Team: Uruguay | TeamID: 36
 PlayerID                     PlayerName Position  JerseyNumber
     1171           Ramón ROCHET ALVAREZ       GK             1
     1172        María GIMÉNEZ DE VARGAS       DF             2
     1173             Enzo CÁCERES RAMOS       DF             3
     1174       Federico ARAÚJO DA SILVA       DF             4
     1175                 UGARTE RIBEIRO       MF             5
     1176               BENTANCUR COLMÁN       MF             6
     1177      Nicolás DE LA CRUZ ARCOSA       MF             7
     1178      Santiago VALVERDE DIPETTA       MF             8
     1179          Gabriel NUÑEZ RIBEIRO       FW             9
     1180 Daniel DE ARRASCAETA BENEDETTI       MF            10
     1181              PELLISTRI REBOLLO       FW            11
     1182          Andres MELE CASTAÑERO       GK            12
     1183                 VARELA OLIVERA       DF            13
     1184              

## 18.22 Targeted Search for the Three Unresolved Scorers

The complete squad output is being truncated.

We will instead search the official Players.csv directly for the
relevant name components:

- Agustín / Cano — Uruguay
- Neymar — Brazil
- Stephan / Eustáquio — Canada

This will show every matching official player record without printing
the full squads.

In [40]:
print("TARGETED SEARCH — OFFICIAL PLAYERS TABLE")
print("=" * 100)

search_terms = {
    "Agustín Cano": ["AGUSTIN", "CANO"],
    "Neymar": ["NEYMAR"],
    "Stephan Eustáquio": ["STEPHAN", "EUSTAQUIO", "EUSTÁQUIO"]
}

for scorer_name, terms in search_terms.items():

    scorer_row = final_unmatched[
        final_unmatched["name"] == scorer_name
    ].iloc[0]

    team_name = scorer_row["TeamForMatching"]
    team_id = team_name_to_id[team_name]

    team_players = players_df[
        players_df["TeamID"] == team_id
    ].copy()

    # Search each term independently
    mask = pd.Series(False, index=team_players.index)

    for term in terms:
        normalized_term = normalize_player_name(term)

        mask = mask | team_players["PlayerName"].apply(
            lambda x: normalized_term in normalize_player_name(x)
        )

    candidates = team_players[
        mask
    ][
        ["PlayerID", "PlayerName", "Position", "JerseyNumber"]
    ].sort_values("JerseyNumber")

    print(
        f"\n{scorer_name} | "
        f"Team: {scorer_row['team']} | "
        f"TeamID: {team_id}"
    )

    if candidates.empty:
        print("  NO NAME COMPONENT MATCH FOUND")
    else:
        print(candidates.to_string(index=False))

TARGETED SEARCH — OFFICIAL PLAYERS TABLE

Agustín Cano | Team: Uruguay | TeamID: 36
 PlayerID PlayerName Position  JerseyNumber
     1184   CANOBBIO       MF            14

Neymar | Team: Brazil | TeamID: 3
  NO NAME COMPONENT MATCH FOUND

Stephan Eustáquio | Team: Canada | TeamID: 16
 PlayerID PlayerName Position  JerseyNumber
      215  EUSTAQUIO       MF             7


## 18.23 Trace Unresolved Goal Events

Agustín Cano and Neymar do not have corresponding records in the
official Players.csv.

We will now locate their exact goal events in the collected goal data,
including MatchID, TeamID, minute, and opposing team.

This will allow us to verify the source records before deciding how
they should be represented in the Goals table.

No PlayerID assignment will be made yet.

In [41]:
print("UNRESOLVED GOAL EVENTS")
print("=" * 100)

unresolved_names = ["Agustín Cano", "Neymar"]

for scorer_name in unresolved_names:

    print(f"\nSCORER: {scorer_name}")
    print("-" * 100)

    # Find all goal events for this scorer
    events = goals_raw_df[
        goals_raw_df["PlayerName"].apply(
            lambda x: normalize_player_name(x)
        ) == normalize_player_name(scorer_name)
    ].copy()

    if events.empty:
        print("No goal events found.")
        continue

    for _, goal in events.iterrows():

        match_id = int(goal["MatchID"])
        team_id = int(goal["TeamID"])

        match = matches_df[
            matches_df["MatchID"] == match_id
        ].iloc[0]

        team1_name = teams_df[
            teams_df["TeamID"] == int(match["Team1ID"])
        ]["TeamName"].iloc[0]

        team2_name = teams_df[
            teams_df["TeamID"] == int(match["Team2ID"])
        ]["TeamName"].iloc[0]

        scoring_team = teams_df[
            teams_df["TeamID"] == team_id
        ]["TeamName"].iloc[0]

        print(
            f"MatchID: {match_id} | "
            f"Date: {match['MatchDate']} | "
            f"Fixture: {team1_name} vs {team2_name} | "
            f"Scoring Team: {scoring_team} | "
            f"Minute: {goal['Minute']} | "
            f"GoalType: {goal['GoalType']}"
        )

UNRESOLVED GOAL EVENTS

SCORER: Agustín Cano
----------------------------------------------------------------------------------------------------
MatchID: 37 | Date: 2026-06-21 | Fixture: Uruguay vs Cabo Verde | Scoring Team: Uruguay | Minute: 45+6 | GoalType: None

SCORER: Neymar
----------------------------------------------------------------------------------------------------
MatchID: 91 | Date: 2026-07-05 | Fixture: Brazil vs Norway | Scoring Team: Brazil | Minute: 90+10 | GoalType: Penalty


## 18.24 Verify the Two Anomalous Goal Records

Two scorer names cannot currently be mapped to the official Players
table.

We will inspect the complete goal-related data returned by the match
source for:

- MatchID 37 — Uruguay vs Cabo Verde
- MatchID 91 — Brazil vs Norway

The purpose is to determine whether the source contains another field
or alternate representation of the scorer that can resolve the
PlayerID mapping.

No PlayerID assignment will be made yet.

In [42]:
print("MATCH-SOURCE VERIFICATION")
print("=" * 100)

# MatchIDs from our database that need investigation
investigation_match_ids = [37, 91]

for our_match_id in investigation_match_ids:

    match_row = matches_df[
        matches_df["MatchID"] == our_match_id
    ].iloc[0]

    team1_id = int(match_row["Team1ID"])
    team2_id = int(match_row["Team2ID"])

    team1_name = teams_df[
        teams_df["TeamID"] == team1_id
    ]["TeamName"].iloc[0]

    team2_name = teams_df[
        teams_df["TeamID"] == team2_id
    ]["TeamName"].iloc[0]

    # Find the source match ID by matching the fixture
    source_match_id = None

    for source_id in range(104):

        url = (
            "https://foot-boll.com/api/data.php"
            f"?action=match&id={source_id}"
        )

        response = requests.get(url, timeout=30)

        if response.status_code != 200:
            continue

        data = response.json().get("match", {})

        source_team1 = data.get("team1", "")
        source_team2 = data.get("team2", "")

        source_team1_id = team_name_to_id.get(
            team_aliases.get(source_team1, source_team1)
        )

        source_team2_id = team_name_to_id.get(
            team_aliases.get(source_team2, source_team2)
        )

        if {
            source_team1_id,
            source_team2_id
        } == {
            team1_id,
            team2_id
        }:
            source_match_id = source_id
            source_match_data = data
            break

    print("\n" + "-" * 100)
    print(
        f"Our MatchID: {our_match_id} | "
        f"{team1_name} vs {team2_name}"
    )
    print("Source MatchID:", source_match_id)

    if source_match_id is None:
        print("SOURCE MATCH NOT FOUND")
        continue

    print("\nSource match keys:")
    print(list(source_match_data.keys()))

    print("\nGoals 1:")
    print(source_match_data.get("goals1", []))

    print("\nGoals 2:")
    print(source_match_data.get("goals2", []))

    print("\nCards:")
    print(source_match_data.get("cards", []))

MATCH-SOURCE VERIFICATION

----------------------------------------------------------------------------------------------------
Our MatchID: 37 | Uruguay vs Cabo Verde
Source MatchID: 45

Source match keys:
['id', 'round', 'group', 'team1', 'team2', 'team1_ar', 'team2_ar', 'flag1', 'flag2', 'status', 'score', 'penalties', 'winner', 'live_minute', 'date', 'time', 'datetime', 'ground', 'ht', 'goals1', 'goals2', 'cards', 'stats']

Goals 1:
[{'name': 'Maxi Araújo', 'minute': '44'}, {'name': 'Agustín Cano', 'minute': '45+6'}]

Goals 2:
[{'name': 'Kevin Pina', 'minute': '21'}, {'name': 'Hélio Varela', 'minute': '61'}]

Cards:
[{'team': 2, 'minute': 5, 'name': 'Sidny Lopes Cabral', 'type': 'yellow', 'reason_ar': 'خطأ قويّ', 'reason_en': 'a bad foul'}, {'team': 1, 'minute': 20, 'name': 'Rodrigo Bentancur', 'type': 'yellow', 'reason_ar': 'خطأ قويّ', 'reason_en': 'a bad foul'}, {'team': 1, 'minute': 58, 'name': 'Mathías Olivera', 'type': 'yellow', 'reason_ar': 'خطأ قويّ', 'reason_en': 'a bad fou

## 18.25 Locate Neymar in the Official Players Table

FIFA confirms that Neymar scored Brazil's penalty against Norway at
90+10 and wore Brazil's number 10.

The current name search did not find "Neymar", so we will inspect the
Brazil player record with JerseyNumber 10 directly.

This will give us the exact PlayerID to use in the Goals table.

In [43]:
print("BRAZIL JERSEY #10")
print("=" * 100)

brazil_team_id = team_name_to_id["Brazil"]

brazil_number_10 = players_df[
    (players_df["TeamID"] == brazil_team_id) &
    (players_df["JerseyNumber"] == 10)
][
    ["PlayerID", "PlayerName", "Position", "JerseyNumber"]
]

print(
    brazil_number_10.to_string(index=False)
)

BRAZIL JERSEY #10
 PlayerID             PlayerName Position  JerseyNumber
      166 DA SILVA SANTOS JÚNIOR       FW            10


## 18.26 Build Complete Goal Scorer → PlayerID Mapping

We have now verified the player-matching method.

For the 173 normally matched scorer records, the mapping is based on:

    normalized scorer name + team
                ↓
        physical player record
                ↓
          jersey number
                ↓
        official Players.csv
                ↓
             PlayerID

Five exceptional scorer names were manually verified against the
official player records:

- Agustín Cano → PlayerID 1184 (Agustín Canobbio)
- Kaan Ayhan → PlayerID 1166
- Leo Østigard → PlayerID 810
- Neymar → PlayerID 166
- Stephan Eustáquio → PlayerID 215

This cell creates the complete scorer-to-PlayerID lookup and validates
that every scorer in the tournament can be mapped to exactly one
official player.

In [44]:
# Build the complete scorer -> PlayerID mapping

print("BUILDING COMPLETE GOAL SCORER → PLAYERID MAPPING")
print("=" * 100)

# ---------------------------------------------------------
# 1. Build physical player -> official PlayerID lookup
#    using Team + JerseyNumber
# ---------------------------------------------------------

physical_to_player_id = {}

for _, physical_player in physical_df.iterrows():

    physical_team = physical_player["TeamForMatching"]
    jersey = int(physical_player["num"])

    team_id = team_name_to_id[physical_team]

    official_player = players_df[
        (players_df["TeamID"] == team_id) &
        (players_df["JerseyNumber"] == jersey)
    ]

    if len(official_player) == 1:

        player_id = int(
            official_player.iloc[0]["PlayerID"]
        )

        physical_to_player_id[
            (physical_player["NameNormalized2"], team_id)
        ] = player_id


# ---------------------------------------------------------
# 2. Match the 173 normally resolved scorer records
# ---------------------------------------------------------

scorer_player_mapping = {}

for _, scorer in scorers_df.iterrows():

    scorer_name = scorer["name"]
    team_name = scorer["TeamForMatching"]
    team_id = team_name_to_id[team_name]

    # Normalized scorer name
    normalized_name = normalize_player_name(scorer_name)

    key = (normalized_name, team_id)

    if key in physical_to_player_id:

        scorer_player_mapping[
            (normalized_name, team_id)
        ] = physical_to_player_id[key]


# ---------------------------------------------------------
# 3. Add the five verified exceptional cases
# ---------------------------------------------------------

special_player_mapping = {
    (
        normalize_player_name("Agustín Cano"),
        team_name_to_id["Uruguay"]
    ): 1184,

    (
        normalize_player_name("Kaan Ayhan"),
        team_name_to_id["Türkiye"]
    ): 1166,

    (
        normalize_player_name("Leo Østigard"),
        team_name_to_id["Norway"]
    ): 810,

    (
        normalize_player_name("Neymar"),
        team_name_to_id["Brazil"]
    ): 166,

    (
        normalize_player_name("Stephan Eustáquio"),
        team_name_to_id["Canada"]
    ): 215
}

scorer_player_mapping.update(
    special_player_mapping
)


# ---------------------------------------------------------
# 4. Apply mapping to every goal event
# ---------------------------------------------------------

goals_with_player_df = goals_raw_df.copy()

goals_with_player_df["PlayerID"] = (
    goals_with_player_df.apply(
        lambda row: scorer_player_mapping.get(
            (
                normalize_player_name(row["PlayerName"]),
                int(row["TeamID"])
            )
        ),
        axis=1
    )
)


# ---------------------------------------------------------
# 5. Validate mapping coverage
# ---------------------------------------------------------

total_goals = len(goals_with_player_df)

mapped_goals = (
    goals_with_player_df["PlayerID"].notna().sum()
)

unmapped_goals = (
    goals_with_player_df["PlayerID"].isna().sum()
)

print("\nMAPPING VALIDATION")
print("-" * 100)

print("Total goal events:", total_goals)
print("Mapped goal events:", mapped_goals)
print("Unmapped goal events:", unmapped_goals)

print(
    "\nExpected: 308 total goals and 308 mapped goals."
)

if unmapped_goals > 0:

    print("\nUNMAPPED GOALS:")
    print(
        goals_with_player_df[
            goals_with_player_df["PlayerID"].isna()
        ][
            [
                "MatchID",
                "TeamID",
                "PlayerName",
                "Minute",
                "GoalType"
            ]
        ].to_string(index=False)
    )

BUILDING COMPLETE GOAL SCORER → PLAYERID MAPPING

MAPPING VALIDATION
----------------------------------------------------------------------------------------------------
Total goal events: 308
Mapped goal events: 294
Unmapped goal events: 14

Expected: 308 total goals and 308 mapped goals.

UNMAPPED GOALS:
 MatchID  TeamID         PlayerName Minute GoalType
       8      44        Miro Muheim   90+4 Own Goal
      27      16      Mohamed Manai     75 Own Goal
      52      26    Mahmoud Abunada     34 Own Goal
      50      39     Yassine Bounou     10 Own Goal
       4      12   Damian Bobadilla      7 Own Goal
      32      12    Cameron Burgess     11 Own Goal
      58       9      Ellyes Skhiri      3 Own Goal
      16       2       Mohamed Hany     66 Own Goal
      38      10 Hassan Al-Tambakti     49 Own Goal
      18      22      Aymen Hussein   90+6 Own Goal
      20      14      Yazan Al-Arab     76 Own Goal
      47      23  Abduvohid Nematov     60 Own Goal
      86       1

## 18.27 Resolve Own Goals

The remaining 14 unmapped goal events are all marked as Own Goal.

For an own goal:

- Goals.TeamID = team credited with the goal
- Goals.PlayerID = player who scored the own goal
- The own-goal player belongs to the opposing team

Therefore, we will search both teams in each match for the scorer's
normalized name.

We will use an exact normalized name + team match where available.
No fuzzy matching will be used.

In [45]:
print("OWN GOAL PLAYER RESOLUTION")
print("=" * 100)

own_goals = goals_with_player_df[
    (goals_with_player_df["GoalType"] == "Own Goal") &
    (goals_with_player_df["PlayerID"].isna())
].copy()

resolved_own_goals = []

for _, goal in own_goals.iterrows():

    match_id = int(goal["MatchID"])
    scorer_name = goal["PlayerName"]

    match = matches_df[
        matches_df["MatchID"] == match_id
    ].iloc[0]

    team1_id = int(match["Team1ID"])
    team2_id = int(match["Team2ID"])

    team1_name = teams_df[
        teams_df["TeamID"] == team1_id
    ]["TeamName"].iloc[0]

    team2_name = teams_df[
        teams_df["TeamID"] == team2_id
    ]["TeamName"].iloc[0]

    normalized_scorer = normalize_player_name(scorer_name)

    # Search both teams in the match
    candidates = players_df[
        players_df["TeamID"].isin([team1_id, team2_id])
    ].copy()

    candidates["NormalizedName"] = candidates[
        "PlayerName"
    ].apply(normalize_player_name)

    exact_candidates = candidates[
        candidates["NormalizedName"] == normalized_scorer
    ]

    print("\n" + "-" * 100)
    print(
        f"MatchID: {match_id} | "
        f"{team1_name} vs {team2_name}"
    )
    print(
        f"Goal credited to TeamID: {int(goal['TeamID'])}"
    )
    print(
        f"Own-goal player from source: {scorer_name} | "
        f"Minute: {goal['Minute']}"
    )

    if len(exact_candidates) == 1:

        candidate = exact_candidates.iloc[0]

        resolved_own_goals.append({
            "MatchID": match_id,
            "PlayerID": int(candidate["PlayerID"]),
            "PlayerName": scorer_name,
            "OfficialPlayerName": candidate["PlayerName"],
            "PlayerTeamID": int(candidate["TeamID"]),
            "GoalTeamID": int(goal["TeamID"]),
            "JerseyNumber": int(candidate["JerseyNumber"]),
            "Minute": goal["Minute"]
        })

        print(
            f"RESOLVED → PlayerID {int(candidate['PlayerID'])} | "
            f"{candidate['PlayerName']} | "
            f"Jersey #{int(candidate['JerseyNumber'])} | "
            f"Player TeamID {int(candidate['TeamID'])}"
        )

    elif len(exact_candidates) == 0:

        print("NO EXACT PLAYER MATCH")

    else:

        print("MULTIPLE EXACT PLAYER MATCHES:")
        print(
            exact_candidates[
                [
                    "PlayerID",
                    "PlayerName",
                    "TeamID",
                    "JerseyNumber"
                ]
            ].to_string(index=False)
        )


print("\n" + "=" * 100)
print(
    "Own goals successfully resolved:",
    len(resolved_own_goals),
    "of",
    len(own_goals)
)

OWN GOAL PLAYER RESOLUTION

----------------------------------------------------------------------------------------------------
MatchID: 8 | Qatar vs Switzerland
Goal credited to TeamID: 44
Own-goal player from source: Miro Muheim | Minute: 90+4
NO EXACT PLAYER MATCH

----------------------------------------------------------------------------------------------------
MatchID: 27 | Canada vs Qatar
Goal credited to TeamID: 16
Own-goal player from source: Mohamed Manai | Minute: 75
NO EXACT PLAYER MATCH

----------------------------------------------------------------------------------------------------
MatchID: 52 | Bosnia and Herzegovina vs Qatar
Goal credited to TeamID: 26
Own-goal player from source: Mahmoud Abunada | Minute: 34
NO EXACT PLAYER MATCH

----------------------------------------------------------------------------------------------------
MatchID: 50 | Morocco vs Haiti
Goal credited to TeamID: 39
Own-goal player from source: Yassine Bounou | Minute: 10
RESOLVED → PlayerID

## 18.28 Search Own-Goal Scorers in Physical Player Data

The 14 remaining unmapped records are all own goals.

The previous search used the official Players table and found only
one exact match.

We will now search the physical player dataset by normalized scorer
name across all teams.

If a scorer is found, the physical record provides the jersey number,
which can then be matched to the official Players table.

The goal-crediting TeamID will remain unchanged.

In [46]:
print("OWN-GOAL SCORER SEARCH — PHYSICAL DATA")
print("=" * 100)

own_goal_search_results = []

for _, goal in own_goals.iterrows():

    scorer_name = goal["PlayerName"]
    normalized_scorer = normalize_player_name(scorer_name)

    # Search the ENTIRE physical dataset by normalized player name
    candidates = physical_df[
        physical_df["NameNormalized2"] == normalized_scorer
    ][
        ["name", "team", "TeamForMatching", "num"]
    ].copy()

    print("\n" + "-" * 100)
    print(
        f"Scorer: {scorer_name} | "
        f"MatchID: {int(goal['MatchID'])} | "
        f"Goal TeamID: {int(goal['TeamID'])} | "
        f"Minute: {goal['Minute']}"
    )

    if candidates.empty:

        print("NO EXACT MATCH IN PHYSICAL DATA")

    else:

        print("PHYSICAL PLAYER MATCH:")
        print(
            candidates.to_string(index=False)
        )

        for _, candidate in candidates.iterrows():

            own_goal_search_results.append({
                "MatchID": int(goal["MatchID"]),
                "ScorerName": scorer_name,
                "GoalTeamID": int(goal["TeamID"]),
                "PlayerTeam": candidate["TeamForMatching"],
                "JerseyNumber": int(candidate["num"]),
                "PhysicalName": candidate["name"]
            })


print("\n" + "=" * 100)

print(
    "Own-goal scorers found in physical data:",
    len(own_goal_search_results)
)

OWN-GOAL SCORER SEARCH — PHYSICAL DATA

----------------------------------------------------------------------------------------------------
Scorer: Miro Muheim | MatchID: 8 | Goal TeamID: 44 | Minute: 90+4
PHYSICAL PLAYER MATCH:
       name        team TeamForMatching  num
Miro MUHEIM Switzerland     Switzerland    2

----------------------------------------------------------------------------------------------------
Scorer: Mohamed Manai | MatchID: 27 | Goal TeamID: 16 | Minute: 75
PHYSICAL PLAYER MATCH:
         name  team TeamForMatching  num
MOHAMED MANAI Qatar           Qatar   26

----------------------------------------------------------------------------------------------------
Scorer: Mahmoud Abunada | MatchID: 52 | Goal TeamID: 26 | Minute: 34
PHYSICAL PLAYER MATCH:
           name  team TeamForMatching  num
MAHMOUD ABUNADA Qatar           Qatar    1

----------------------------------------------------------------------------------------------------
Scorer: Yassine Bounou |

## 18.29 Resolve All Own-Goal Players to PlayerID

All 14 own-goal scorers were found in the physical player dataset.

We will now use each player's TeamID and JerseyNumber to locate the
corresponding official record in Players.csv.

Important:

- GoalTeamID remains the team credited with the goal.
- PlayerID belongs to the player who scored the own goal.
- Therefore, for an own goal, PlayerID's TeamID may differ from
  GoalTeamID.

We will validate that every own-goal scorer maps to exactly one
official PlayerID.

In [47]:
print("RESOLVING ALL OWN GOALS → OFFICIAL PLAYERID")
print("=" * 100)

own_goal_player_mappings = []

for result in own_goal_search_results:

    player_team = result["PlayerTeam"]
    jersey = result["JerseyNumber"]

    player_team_id = team_name_to_id[player_team]

    official_player = players_df[
        (players_df["TeamID"] == player_team_id) &
        (players_df["JerseyNumber"] == jersey)
    ]

    if len(official_player) != 1:

        print(
            f"ERROR: {result['ScorerName']} | "
            f"{player_team} | #{jersey} | "
            f"Official matches: {len(official_player)}"
        )

        continue

    player = official_player.iloc[0]

    own_goal_player_mappings.append({
        "MatchID": result["MatchID"],
        "ScorerName": result["ScorerName"],
        "GoalTeamID": result["GoalTeamID"],
        "PlayerTeamID": player_team_id,
        "PlayerID": int(player["PlayerID"]),
        "OfficialPlayerName": player["PlayerName"],
        "JerseyNumber": jersey,
        "Minute": goals_raw_df[
            (goals_raw_df["MatchID"] == result["MatchID"]) &
            (goals_raw_df["PlayerName"] == result["ScorerName"])
        ]["Minute"].iloc[0]
    })


own_goal_mapping_df = pd.DataFrame(
    own_goal_player_mappings
)

print("\nRESOLVED OWN GOALS")
print("-" * 100)

print(
    own_goal_mapping_df[
        [
            "MatchID",
            "ScorerName",
            "GoalTeamID",
            "PlayerTeamID",
            "PlayerID",
            "OfficialPlayerName",
            "JerseyNumber",
            "Minute"
        ]
    ].to_string(index=False)
)

print("\n" + "=" * 100)

print(
    "Own goals expected:",
    len(own_goals)
)

print(
    "Own goals resolved:",
    len(own_goal_mapping_df)
)

print(
    "Unique PlayerIDs:",
    own_goal_mapping_df["PlayerID"].nunique()
)

print(
    "Missing PlayerIDs:",
    own_goal_mapping_df["PlayerID"].isna().sum()
)

print(
    "PlayerID/Team/Jersey validation:",
    (
        own_goal_mapping_df.apply(
            lambda row: len(
                players_df[
                    (players_df["PlayerID"] == row["PlayerID"]) &
                    (players_df["TeamID"] == row["PlayerTeamID"]) &
                    (players_df["JerseyNumber"] == row["JerseyNumber"])
                ]
            ) == 1,
            axis=1
        ).all()
    )
)

RESOLVING ALL OWN GOALS → OFFICIAL PLAYERID

RESOLVED OWN GOALS
----------------------------------------------------------------------------------------------------
 MatchID         ScorerName  GoalTeamID  PlayerTeamID  PlayerID              OfficialPlayerName  JerseyNumber Minute
       8        Miro Muheim          44            11      1094                Max Maria MUHEIM             2   90+4
      27      Mohamed Manai          16            44       936            Mohamed Naceur MANAI            26     75
      52    Mahmoud Abunada          26            44       911       Mahmoud Ibrahim M ABUNADA             1     34
      50     Yassine Bounou          39            21       729                  Yassine BOUNOU             1     10
       4   Damian Bobadilla          12            32       874  Damián Josue BOBADILLA BENITEZ            16      7
      32    Cameron Burgess          12            13        73  Cameron Robert BURGESS BURGESS            21     11
      58      El

## 18.30 Diagnose Remaining Unmapped Goals

The final PlayerID conversion failed because some goal events still
have no PlayerID.

Before changing the mapping logic, we will identify the exact
remaining unmapped goal events.

No data will be exported until all 308 goal events have a verified
PlayerID.

In [50]:
print("REMAINING UNMAPPED GOALS")
print("=" * 100)

# Check how many PlayerIDs are still missing
missing_mask = final_goals_df["PlayerID"].isna()

print("Total goal events:", len(final_goals_df))
print("Mapped:", (~missing_mask).sum())
print("Unmapped:", missing_mask.sum())

print("\nUnmapped goal events:")
print("-" * 100)

print(
    final_goals_df.loc[
        missing_mask,
        [
            "MatchID",
            "TeamID",
            "PlayerName",
            "Minute",
            "GoalType"
        ]
    ].to_string(index=False)
)

REMAINING UNMAPPED GOALS
Total goal events: 308
Mapped: 307
Unmapped: 1

Unmapped goal events:
----------------------------------------------------------------------------------------------------
 MatchID  TeamID    PlayerName Minute GoalType
      18      22 Aymen Hussein   90+6 Own Goal


## 18.30 Fix the Final Unmapped Goal

Only one of the 308 goal events remains unmapped.

The event is:

- MatchID: 18
- Source scorer: Aymen Hussein
- Minute: 90+6
- GoalType: Own Goal

This player was already independently verified in the physical
dataset and official Players.csv:

- Physical player: Hussein Ghadhban GHADHBAN
- JerseyNumber: 18
- PlayerID: 616

We will apply this verified PlayerID to this specific goal event.

In [51]:
# Apply the already verified PlayerID for Aymen Hussein's own goal

mask = (
    (final_goals_df["MatchID"] == 18) &
    (final_goals_df["TeamID"] == 22) &
    (final_goals_df["PlayerName"] == "Aymen Hussein") &
    (final_goals_df["Minute"] == "90+6") &
    (final_goals_df["GoalType"] == "Own Goal")
)

print("Matching rows found:", mask.sum())

final_goals_df.loc[mask, "PlayerID"] = 616

print("\nUpdated row:")
print(
    final_goals_df.loc[
        mask,
        [
            "MatchID",
            "TeamID",
            "PlayerName",
            "Minute",
            "GoalType",
            "PlayerID"
        ]
    ].to_string(index=False)
)

print("\nRemaining unmapped goals:")
print(
    final_goals_df["PlayerID"].isna().sum()
)

Matching rows found: 1

Updated row:
 MatchID  TeamID    PlayerName Minute GoalType  PlayerID
      18      22 Aymen Hussein   90+6 Own Goal     616.0

Remaining unmapped goals:
0


## 18.31 Inspect Final Goals Columns

The goal data contains 308 rows with no missing MatchID, PlayerID,
TeamID, PlayerName, or Minute.

However, the column validation returned False and the duplicate check
failed because an extra column contains list-valued data.

We will inspect the exact columns and data types before selecting the
final SQL-ready columns.

In [53]:
print("FINAL GOALS DATAFRAME STRUCTURE")
print("=" * 100)

print("Columns:")
print(final_goals_df.columns.tolist())

print("\nData types:")
print(final_goals_df.dtypes)

print("\nFirst 5 rows:")
print(
    final_goals_df.head().to_string()
)

print("\nColumns containing list values:")
for column in final_goals_df.columns:

    if final_goals_df[column].apply(
        lambda x: isinstance(x, list)
    ).any():

        print(
            f"  {column}: contains list values"
        )

FINAL GOALS DATAFRAME STRUCTURE
Columns:
['SourceMatchID', 'MatchID', 'TeamID', 'PlayerName', 'Minute', 'GoalType', 'PlayerID', 'SourceLastName', 'CandidateCount', 'NormalizedSourceName', 'Candidates', 'BestCandidates']

Data types:
SourceMatchID             int64
MatchID                   int64
TeamID                    int64
PlayerName               object
Minute                   object
GoalType                 object
PlayerID                float64
SourceLastName           object
CandidateCount            int64
NormalizedSourceName     object
Candidates               object
BestCandidates           object
dtype: object

First 5 rows:
   SourceMatchID  MatchID  TeamID       PlayerName Minute GoalType  PlayerID SourceLastName  CandidateCount NormalizedSourceName                                                                                                                                                                                            Candidates                            

## 18.32 Create the SQL-Ready Goals DataFrame

The working dataframe contains temporary matching and investigation
columns that are not part of the SQL Goals table.

We will now create a clean dataframe containing exactly the fields
required by the database:

- GoalID
- MatchID
- PlayerID
- TeamID
- Minute
- GoalType

GoalID will be assigned sequentially from 1 to 308.

The original working dataframe will not be modified.

In [54]:
print("CREATING SQL-READY GOALS DATAFRAME")
print("=" * 100)

# Create clean SQL-ready dataframe
goals_df = final_goals_df[
    [
        "MatchID",
        "PlayerID",
        "TeamID",
        "Minute",
        "GoalType"
    ]
].copy()

# Ensure integer foreign keys
goals_df["MatchID"] = goals_df["MatchID"].astype(int)
goals_df["PlayerID"] = goals_df["PlayerID"].astype(int)
goals_df["TeamID"] = goals_df["TeamID"].astype(int)

# GoalID is the primary key
goals_df.insert(
    0,
    "GoalID",
    range(1, len(goals_df) + 1)
)

# Reorder explicitly
goals_df = goals_df[
    [
        "GoalID",
        "MatchID",
        "PlayerID",
        "TeamID",
        "Minute",
        "GoalType"
    ]
]

print("Columns:")
print(goals_df.columns.tolist())

print("\nShape:")
print(goals_df.shape)

print("\nData types:")
print(goals_df.dtypes)

print("\nFirst 10 rows:")
print(
    goals_df.head(10).to_string(index=False)
)

CREATING SQL-READY GOALS DATAFRAME
Columns:
['GoalID', 'MatchID', 'PlayerID', 'TeamID', 'Minute', 'GoalType']

Shape:
(308, 6)

Data types:
GoalID       int64
MatchID      int64
PlayerID     int64
TeamID       int64
Minute      object
GoalType    object
dtype: object

First 10 rows:
 GoalID  MatchID  PlayerID  TeamID Minute GoalType
      1        1       718       8      9     None
      2        1       711       8     67     None
      3        2       682      31     67     None
      4        2       694      31     80     None
      5        2       371      38     59     None
      6       25       382      38      6     None
      7       25      1018      24     83  Penalty
      8       28       709       8     50     None
      9       53       722       8     55     None
     10       53       718       8     61     None


## 18.33 Final Relational Validation

The SQL-ready Goals dataframe contains 308 records.

We will now verify:

- GoalID is unique and sequential.
- MatchID exists in Matches.
- PlayerID exists in Players.
- TeamID exists in Teams.
- MatchID + PlayerID + TeamID relationships are valid.
- There are exactly 308 goal records.
- GoalType contains only NULL, Penalty, or Own Goal.
- There are 16 penalty goals and 14 own goals.

This is the final validation before exporting Goals.csv.

In [55]:
print("FINAL RELATIONAL VALIDATION — GOALS")
print("=" * 100)

# ---------------------------------------------------------
# Basic validation
# ---------------------------------------------------------

print("Total rows:", len(goals_df))
print("Expected rows: 308")

print(
    "GoalID duplicates:",
    goals_df["GoalID"].duplicated().sum()
)

print(
    "GoalID sequence correct:",
    goals_df["GoalID"].tolist() == list(
        range(1, len(goals_df) + 1)
    )
)

print(
    "Missing values:",
    goals_df.isna().sum().sum()
)

# ---------------------------------------------------------
# Foreign keys
# ---------------------------------------------------------

invalid_match_ids = set(
    goals_df["MatchID"]
) - set(
    matches_df["MatchID"]
)

invalid_player_ids = set(
    goals_df["PlayerID"]
) - set(
    players_df["PlayerID"]
)

invalid_team_ids = set(
    goals_df["TeamID"]
) - set(
    teams_df["TeamID"]
)

print("\nFOREIGN KEY VALIDATION")
print("-" * 100)

print(
    "Invalid MatchIDs:",
    len(invalid_match_ids)
)

print(
    "Invalid PlayerIDs:",
    len(invalid_player_ids)
)

print(
    "Invalid TeamIDs:",
    len(invalid_team_ids)
)

# ---------------------------------------------------------
# GoalType validation
# ---------------------------------------------------------

valid_goal_types = {
    "Penalty",
    "Own Goal"
}

invalid_goal_types = set(
    goals_df["GoalType"].dropna()
) - valid_goal_types

print("\nGOAL TYPE VALIDATION")
print("-" * 100)

print(
    "Invalid GoalTypes:",
    len(invalid_goal_types)
)

print(
    "\nGoal type distribution:"
)

print(
    goals_df["GoalType"]
    .fillna("Normal")
    .value_counts()
    .sort_index()
    .to_string()
)

# ---------------------------------------------------------
# Special goal counts
# ---------------------------------------------------------

normal_goals = (
    goals_df["GoalType"].isna().sum()
)

penalty_goals = (
    (goals_df["GoalType"] == "Penalty").sum()
)

own_goals = (
    (goals_df["GoalType"] == "Own Goal").sum()
)

print("\nGOAL COUNTS")
print("-" * 100)

print("Normal goals:", normal_goals)
print("Penalty goals:", penalty_goals)
print("Own goals:", own_goals)
print("Total:", normal_goals + penalty_goals + own_goals)

# ---------------------------------------------------------
# Player-Team relationship validation
# ---------------------------------------------------------

player_team_check = goals_df.merge(
    players_df[
        ["PlayerID", "TeamID"]
    ].rename(
        columns={"TeamID": "PlayerTeamID"}
    ),
    on="PlayerID",
    how="left"
)

invalid_player_team = (
    (
        player_team_check["GoalType"] != "Own Goal"
    )
    &
    (
        player_team_check["TeamID"]
        != player_team_check["PlayerTeamID"]
    )
).sum()

print("\nPLAYER / GOAL TEAM VALIDATION")
print("-" * 100)

print(
    "Normal goals where PlayerTeamID != GoalTeamID:",
    invalid_player_team
)

# Own goals are expected to have different player/goal teams
own_goal_team_check = player_team_check[
    player_team_check["GoalType"] == "Own Goal"
]

own_goal_same_team = (
    own_goal_team_check["TeamID"]
    ==
    own_goal_team_check["PlayerTeamID"]
).sum()

print(
    "Own goals where PlayerTeamID == GoalTeamID:",
    own_goal_same_team
)

# ---------------------------------------------------------
# Final pass/fail
# ---------------------------------------------------------

validation_passed = (
    len(goals_df) == 308
    and goals_df["GoalID"].duplicated().sum() == 0
    and goals_df["GoalID"].tolist()
        == list(range(1, 309))
    and goals_df.isna().sum().sum() == 0
    and len(invalid_match_ids) == 0
    and len(invalid_player_ids) == 0
    and len(invalid_team_ids) == 0
    and len(invalid_goal_types) == 0
    and normal_goals == 278
    and penalty_goals == 16
    and own_goals == 14
    and invalid_player_team == 0
)

print("\n" + "=" * 100)

if validation_passed:
    print("FINAL RELATIONAL VALIDATION: PASSED")
else:
    print("FINAL RELATIONAL VALIDATION: FAILED")

FINAL RELATIONAL VALIDATION — GOALS
Total rows: 308
Expected rows: 308
GoalID duplicates: 0
GoalID sequence correct: True
Missing values: 278

FOREIGN KEY VALIDATION
----------------------------------------------------------------------------------------------------
Invalid MatchIDs: 0
Invalid PlayerIDs: 0
Invalid TeamIDs: 0

GOAL TYPE VALIDATION
----------------------------------------------------------------------------------------------------
Invalid GoalTypes: 0

Goal type distribution:
GoalType
Normal      278
Own Goal     14
Penalty      16

GOAL COUNTS
----------------------------------------------------------------------------------------------------
Normal goals: 278
Penalty goals: 16
Own goals: 14
Total: 308

PLAYER / GOAL TEAM VALIDATION
----------------------------------------------------------------------------------------------------
Normal goals where PlayerTeamID != GoalTeamID: 0
Own goals where PlayerTeamID == GoalTeamID: 0

FINAL RELATIONAL VALIDATION: FAILED


## 18.31 Correct Final Goals Validation

The previous validation reported 278 missing values because the
GoalType column is intentionally NULL for normal goals.

This is valid according to the Goals SQL schema because GoalType is
nullable.

We will therefore validate required fields separately from GoalType:

Required:
- GoalID
- MatchID
- PlayerID
- TeamID
- Minute

Optional:
- GoalType

GoalType may contain NULL, Penalty, or Own Goal.

In [56]:
print("CORRECTED FINAL RELATIONAL VALIDATION — GOALS")
print("=" * 100)

# Required columns
required_columns = [
    "GoalID",
    "MatchID",
    "PlayerID",
    "TeamID",
    "Minute",
    "GoalType"
]

print(
    "Columns correct:",
    goals_df.columns.tolist() == required_columns
)

print(
    "Total rows:",
    len(goals_df)
)

# ---------------------------------------------------------
# Required fields
# ---------------------------------------------------------

required_fields = [
    "GoalID",
    "MatchID",
    "PlayerID",
    "TeamID",
    "Minute"
]

print("\nREQUIRED FIELD VALIDATION")
print("-" * 100)

required_missing = goals_df[
    required_fields
].isna().sum()

print(required_missing.to_string())

print(
    "Total missing required values:",
    required_missing.sum()
)

# GoalType is intentionally nullable
print(
    "GoalType NULL values:",
    goals_df["GoalType"].isna().sum()
)

# ---------------------------------------------------------
# GoalID
# ---------------------------------------------------------

print("\nPRIMARY KEY VALIDATION")
print("-" * 100)

print(
    "Duplicate GoalIDs:",
    goals_df["GoalID"].duplicated().sum()
)

print(
    "GoalID sequence correct:",
    goals_df["GoalID"].tolist()
    == list(range(1, 309))
)

# ---------------------------------------------------------
# Foreign keys
# ---------------------------------------------------------

invalid_match_ids = set(
    goals_df["MatchID"]
) - set(
    matches_df["MatchID"]
)

invalid_player_ids = set(
    goals_df["PlayerID"]
) - set(
    players_df["PlayerID"]
)

invalid_team_ids = set(
    goals_df["TeamID"]
) - set(
    teams_df["TeamID"]
)

print("\nFOREIGN KEY VALIDATION")
print("-" * 100)

print("Invalid MatchIDs:", len(invalid_match_ids))
print("Invalid PlayerIDs:", len(invalid_player_ids))
print("Invalid TeamIDs:", len(invalid_team_ids))

# ---------------------------------------------------------
# GoalType
# ---------------------------------------------------------

invalid_goal_types = set(
    goals_df["GoalType"].dropna()
) - {
    "Penalty",
    "Own Goal"
}

print("\nGOAL TYPE VALIDATION")
print("-" * 100)

print(
    "Invalid GoalTypes:",
    len(invalid_goal_types)
)

print(
    goals_df["GoalType"]
    .fillna("Normal")
    .value_counts()
    .sort_index()
    .to_string()
)

# ---------------------------------------------------------
# Goal counts
# ---------------------------------------------------------

normal_goals = goals_df["GoalType"].isna().sum()
penalty_goals = (
    goals_df["GoalType"] == "Penalty"
).sum()
own_goals = (
    goals_df["GoalType"] == "Own Goal"
).sum()

print("\nGOAL COUNTS")
print("-" * 100)

print("Normal goals:", normal_goals)
print("Penalty goals:", penalty_goals)
print("Own goals:", own_goals)
print(
    "Total:",
    normal_goals + penalty_goals + own_goals
)

# ---------------------------------------------------------
# Player-Team relationship
# ---------------------------------------------------------

player_team_check = goals_df.merge(
    players_df[
        ["PlayerID", "TeamID"]
    ].rename(
        columns={"TeamID": "PlayerTeamID"}
    ),
    on="PlayerID",
    how="left"
)

invalid_normal_team = (
    (
        player_team_check["GoalType"].isna()
        |
        (player_team_check["GoalType"] == "Penalty")
    )
    &
    (
        player_team_check["TeamID"]
        != player_team_check["PlayerTeamID"]
    )
).sum()

own_goal_same_team = (
    player_team_check[
        player_team_check["GoalType"] == "Own Goal"
    ]["TeamID"]
    ==
    player_team_check[
        player_team_check["GoalType"] == "Own Goal"
    ]["PlayerTeamID"]
).sum()

print("\nPLAYER / GOAL TEAM VALIDATION")
print("-" * 100)

print(
    "Normal/Penalty goals with wrong PlayerTeamID:",
    invalid_normal_team
)

print(
    "Own goals with same PlayerTeamID as GoalTeamID:",
    own_goal_same_team
)

# ---------------------------------------------------------
# Final result
# ---------------------------------------------------------

validation_passed = (
    goals_df.columns.tolist() == required_columns
    and len(goals_df) == 308
    and required_missing.sum() == 0
    and goals_df["GoalID"].duplicated().sum() == 0
    and goals_df["GoalID"].tolist()
        == list(range(1, 309))
    and len(invalid_match_ids) == 0
    and len(invalid_player_ids) == 0
    and len(invalid_team_ids) == 0
    and len(invalid_goal_types) == 0
    and normal_goals == 278
    and penalty_goals == 16
    and own_goals == 14
    and invalid_normal_team == 0
    and own_goal_same_team == 0
)

print("\n" + "=" * 100)

if validation_passed:
    print("FINAL RELATIONAL VALIDATION: PASSED")
else:
    print("FINAL RELATIONAL VALIDATION: FAILED")

CORRECTED FINAL RELATIONAL VALIDATION — GOALS
Columns correct: True
Total rows: 308

REQUIRED FIELD VALIDATION
----------------------------------------------------------------------------------------------------
GoalID      0
MatchID     0
PlayerID    0
TeamID      0
Minute      0
Total missing required values: 0
GoalType NULL values: 278

PRIMARY KEY VALIDATION
----------------------------------------------------------------------------------------------------
Duplicate GoalIDs: 0
GoalID sequence correct: True

FOREIGN KEY VALIDATION
----------------------------------------------------------------------------------------------------
Invalid MatchIDs: 0
Invalid PlayerIDs: 0
Invalid TeamIDs: 0

GOAL TYPE VALIDATION
----------------------------------------------------------------------------------------------------
Invalid GoalTypes: 0
GoalType
Normal      278
Own Goal     14
Penalty      16

GOAL COUNTS
------------------------------------------------------------------------------------

## 18.34 Export Goals.csv

The Goals dataframe has passed all final validation checks:

- 308 goal events
- 308 valid GoalIDs
- 308 valid PlayerIDs
- 308 valid MatchIDs
- 308 valid TeamIDs
- 278 normal goals
- 16 penalty goals
- 14 own goals
- 0 missing required values
- 0 invalid foreign keys
- 0 invalid GoalTypes

We will now export the final SQL-ready Goals.csv file to the
project Data folder.

In [57]:
from pathlib import Path

# Project Data folder
data_folder = Path.cwd().parent / "Data"

# Output path
goals_path = data_folder / "Goals.csv"

# Export
goals_df.to_csv(
    goals_path,
    index=False,
    encoding="utf-8-sig"
)

print("GOALS CSV EXPORTED")
print("=" * 100)

print("File:", goals_path)
print("Exists:", goals_path.exists())
print("Rows:", len(goals_df))
print("Columns:", len(goals_df.columns))

print("\nColumns:")
print(goals_df.columns.tolist())

GOALS CSV EXPORTED
File: d:\ai course hw\FIFA_WorldCup_2026_Project\Data\Goals.csv
Exists: True
Rows: 308
Columns: 6

Columns:
['GoalID', 'MatchID', 'PlayerID', 'TeamID', 'Minute', 'GoalType']


## 18.35 Reload and Verify Goals.csv

The final Goals.csv file has been exported.

We will now reload the actual file from disk and verify:

- 308 rows
- 6 columns
- correct column names
- no missing required values
- valid GoalIDs
- valid MatchIDs
- valid PlayerIDs
- valid TeamIDs
- correct goal-type counts

This confirms that the exported CSV can be safely used for SQL import.

In [58]:
print("RELOAD AND VERIFY Goals.csv")
print("=" * 100)

# Reload the actual exported CSV
goals_check_df = pd.read_csv(
    goals_path,
    encoding="utf-8-sig"
)

print("Shape:", goals_check_df.shape)

print("\nColumns:")
print(goals_check_df.columns.tolist())

print("\nFirst 10 rows:")
print(
    goals_check_df.head(10).to_string(index=False)
)

# Required columns
expected_columns = [
    "GoalID",
    "MatchID",
    "PlayerID",
    "TeamID",
    "Minute",
    "GoalType"
]

print("\nColumn validation:")
print(
    goals_check_df.columns.tolist()
    == expected_columns
)

# Required fields
required_fields = [
    "GoalID",
    "MatchID",
    "PlayerID",
    "TeamID",
    "Minute"
]

print("\nMissing required values:")
print(
    goals_check_df[required_fields]
    .isna()
    .sum()
    .to_string()
)

# Primary key
print(
    "\nDuplicate GoalIDs:",
    goals_check_df["GoalID"].duplicated().sum()
)

print(
    "GoalID range:",
    goals_check_df["GoalID"].min(),
    "to",
    goals_check_df["GoalID"].max()
)

# Foreign keys
invalid_match_ids = set(
    goals_check_df["MatchID"]
) - set(
    matches_df["MatchID"]
)

invalid_player_ids = set(
    goals_check_df["PlayerID"]
) - set(
    players_df["PlayerID"]
)

invalid_team_ids = set(
    goals_check_df["TeamID"]
) - set(
    teams_df["TeamID"]
)

print("\nForeign key validation:")
print("Invalid MatchIDs:", len(invalid_match_ids))
print("Invalid PlayerIDs:", len(invalid_player_ids))
print("Invalid TeamIDs:", len(invalid_team_ids))

# Goal types
print("\nGoal type counts:")
print(
    goals_check_df["GoalType"]
    .fillna("Normal")
    .value_counts()
    .sort_index()
    .to_string()
)

# Final reload validation
reload_passed = (
    goals_check_df.shape == (308, 6)
    and goals_check_df.columns.tolist()
        == expected_columns
    and goals_check_df[required_fields]
        .isna()
        .sum()
        .sum() == 0
    and goals_check_df["GoalID"]
        .duplicated()
        .sum() == 0
    and goals_check_df["GoalID"].min() == 1
    and goals_check_df["GoalID"].max() == 308
    and len(invalid_match_ids) == 0
    and len(invalid_player_ids) == 0
    and len(invalid_team_ids) == 0
)

print("\n" + "=" * 100)

if reload_passed:
    print("Goals.csv RELOAD VALIDATION: PASSED")
else:
    print("Goals.csv RELOAD VALIDATION: FAILED")

RELOAD AND VERIFY Goals.csv
Shape: (308, 6)

Columns:
['GoalID', 'MatchID', 'PlayerID', 'TeamID', 'Minute', 'GoalType']

First 10 rows:
 GoalID  MatchID  PlayerID  TeamID Minute GoalType
      1        1       718       8      9      NaN
      2        1       711       8     67      NaN
      3        2       682      31     67      NaN
      4        2       694      31     80      NaN
      5        2       371      38     59      NaN
      6       25       382      38      6      NaN
      7       25      1018      24     83  Penalty
      8       28       709       8     50      NaN
      9       53       722       8     55      NaN
     10       53       718       8     61      NaN

Column validation:
True

Missing required values:
GoalID      0
MatchID     0
PlayerID    0
TeamID      0
Minute      0

Duplicate GoalIDs: 0
GoalID range: 1 to 308

Foreign key validation:
Invalid MatchIDs: 0
Invalid PlayerIDs: 0
Invalid TeamIDs: 0

Goal type counts:
GoalType
Normal      278
Own Goal

## 19.1 Verify Final Project Datasets

All data-collection notebooks are complete.

Before starting the SQL database phase, we will verify that all required
CSV files exist in the project's Data folder:

- Teams.csv
- Stadiums.csv
- Matches.csv
- MatchStatistics.csv
- Players.csv
- PlayerPhysicalStats.csv
- Goals.csv

This ensures that the complete dataset is available before importing it
into SQL Server.

In [59]:
from pathlib import Path

data_folder = Path.cwd().parent / "Data"

required_files = [
    "Teams.csv",
    "Stadiums.csv",
    "Matches.csv",
    "MatchStatistics.csv",
    "Players.csv",
    "PlayerPhysicalStats.csv",
    "Goals.csv"
]

print("FINAL DATASET FILE CHECK")
print("=" * 100)

all_exist = True

for filename in required_files:
    file_path = data_folder / filename
    exists = file_path.exists()

    print(f"{filename:<30} Exists: {exists}")

    if not exists:
        all_exist = False

print("\n" + "=" * 100)

if all_exist:
    print("ALL 7 DATASETS FOUND — PASSED")
else:
    print("DATASET FILE CHECK — FAILED")

FINAL DATASET FILE CHECK
Teams.csv                      Exists: True
Stadiums.csv                   Exists: True
Matches.csv                    Exists: True
MatchStatistics.csv            Exists: True
Players.csv                    Exists: True
PlayerPhysicalStats.csv        Exists: True
Goals.csv                      Exists: True

ALL 7 DATASETS FOUND — PASSED


## 28.1 Verify Goals CSV

Reload the Goals CSV and verify its structure, primary key,
foreign keys, required fields, and GoalType values.

In [7]:
import pandas as pd
from pathlib import Path

data_folder = Path.cwd().parent / "Data"

goals_path = data_folder / "Goals.csv"
matches_path = data_folder / "Matches.csv"
players_path = data_folder / "Players.csv"
teams_path = data_folder / "Teams.csv"

# Load datasets
goals_df = pd.read_csv(goals_path)
matches_df = pd.read_csv(matches_path)
players_df = pd.read_csv(players_path)
teams_df = pd.read_csv(teams_path)

print("GOALS CSV VERIFICATION")
print("=" * 100)

print("Shape:", goals_df.shape)

print("\nColumns:")
print(goals_df.columns.tolist())

print("\nMissing values:")
print(goals_df.isna().sum().to_string())

print("\nDuplicate GoalIDs:",
      goals_df["GoalID"].duplicated().sum())

print(
    "\nGoalID range:",
    goals_df["GoalID"].min(),
    "to",
    goals_df["GoalID"].max()
)

print("\nInvalid MatchIDs:",
      (~goals_df["MatchID"].isin(matches_df["MatchID"])).sum())

print("\nInvalid PlayerIDs:",
      (~goals_df["PlayerID"].isin(players_df["PlayerID"])).sum())

print("\nInvalid TeamIDs:",
      (~goals_df["TeamID"].isin(teams_df["TeamID"])).sum())

print("\nGoalType values:")
print(goals_df["GoalType"].value_counts(dropna=False).to_string())

print("\nMinute values missing:",
      goals_df["Minute"].isna().sum())

print("\nFinal checks:")
print("308 rows:", len(goals_df) == 308)
print("6 columns:", len(goals_df.columns) == 6)
print("GoalIDs unique:", goals_df["GoalID"].is_unique)
print("No missing required fields:",
      goals_df[["GoalID", "MatchID", "PlayerID", "TeamID", "Minute"]]
      .isna().sum().sum() == 0)

GOALS CSV VERIFICATION
Shape: (308, 6)

Columns:
['GoalID', 'MatchID', 'PlayerID', 'TeamID', 'Minute', 'GoalType']

Missing values:
GoalID        0
MatchID       0
PlayerID      0
TeamID        0
Minute        0
GoalType    278

Duplicate GoalIDs: 0

GoalID range: 1 to 308

Invalid MatchIDs: 0

Invalid PlayerIDs: 0

Invalid TeamIDs: 0

GoalType values:
GoalType
NaN         278
Penalty      16
Own Goal     14

Minute values missing: 0

Final checks:
308 rows: True
6 columns: True
GoalIDs unique: True
No missing required fields: True


## 28.2 Verify GoalType Distribution

Confirm the number of normal goals, penalty goals, and own goals
in the final Goals dataset.

In [8]:
print("GOAL TYPE DISTRIBUTION")
print("=" * 100)

goal_type_counts = goals_df["GoalType"].fillna("Normal").value_counts()

print(goal_type_counts.to_string())

print("\nExpected:")
print("Normal goals:", (goals_df["GoalType"].isna()).sum())
print("Penalty goals:", (goals_df["GoalType"] == "Penalty").sum())
print("Own goals:", (goals_df["GoalType"] == "Own Goal").sum())

print("\nTotal goals:",
      goal_type_counts.sum())

GOAL TYPE DISTRIBUTION
GoalType
Normal      278
Penalty      16
Own Goal     14

Expected:
Normal goals: 278
Penalty goals: 16
Own goals: 14

Total goals: 308
